# 🏥 BCMID — Data & Quality Agent (Agent 2)

**Projet** : Multimodal Medical AI Automation System with XAI

**Dataset** : BCMID (Breast Cancer Multimodal Imaging Dataset) — Zenodo, DOI 10.5281/zenodo.19402690

Ce notebook implémente le **Data & Quality Agent**, le deuxième maillon du pipeline d'IA médicale multimodale.
Son rôle est de transformer les données brutes (images mammographiques, échographiques et rapports PDF)
en un dataset **propre, validé, anonymisé et prêt pour l'entraînement** de l'Agent Vision (CNN/ViT + Grad-CAM).

## Pipeline global
```
1) Input Layer → 2) Data & Quality Agent (CE NOTEBOOK) → 3) Multimodal Vision Agent
→ 4) Clinical RAG Agent → 5) Reasoning Agent → 6) Quality/Safety Agent → 7) Rapport final
```

## Table des matières

| # | Module | Description |
|---|---|---|
| 1 | `config.py` | Configuration centralisée (chemins, paramètres, seuils) |
| 2 | `zip_io.py` | Lecture et scan du ZIP sans extraction complète |
| 3 | `profiling.py` | Statistiques du dataset (tailles, formats, distribution BI-RADS) |
| 4 | `integrity.py` | Vérification d'intégrité (corrompues, uniformes, doublons) |
| 5 | `modality.py` | Classification par modalité (complet / mammo_seule / echo_seule) |
| 6 | `validation.py` | Jointure images ↔ labels BI-RADS, détection d'anomalies |
| 7 | `preprocess.py` | Normalisation, débruitage, CLAHE, redimensionnement 512×512 |
| 8 | `anonymize.py` | Pseudonymisation SHA-256 + sel secret |
| 9 | `reports.py` | Extraction PDF, nettoyage PII, détection label leakage |
| 10 | `splits.py` | Découpage train/val/test stratifié par patient |
| 11 | `report.py` | Génération du rapport de qualité (JSON + Markdown) |
| 12 | `plots.py` | Visualisations (distribution, avant/après, tailles) |
| 13 | `cli.py` | Orchestration complète du pipeline |
| 14 | Tests | 37 tests pytest sur données synthétiques |


---
## 0. Préparation de l'environnement

On crée les dossiers nécessaires et on installe les dépendances manquantes.


In [ ]:
!mkdir -p dq_agent!mkdir -p tests!pip install -q pdfplumber python-dotenv opencv-python-headless tqdm pyyaml matplotlib scikit-learn pytest

---
## Verification GPU

Cette cellule verifie que Kaggle a bien active le GPU (T4 ou P100).
Si tu vois `CPU only`, va dans `Settings > Accelerator > GPU T4 x2` et relance.

> Le GPU n'est pas obligatoire pour ce pipeline (preprocessing CPU), mais il
> accelere significativement le traitement des 2001 images (~30min vs ~2h).


In [ ]:
# ============================================================
# VERIFICATION GPU (comme notebook1)
# ============================================================

import subprocess
import sys

# 1. nvidia-smi (disponible sur Kaggle si GPU active)
print("=" * 60)
print("VERIFICATION GPU")
print("=" * 60)

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        gpu_info = result.stdout.strip()
        print(f"GPU detecte  : {gpu_info}")
        GPU_AVAILABLE = True
    else:
        print("GPU          : non disponible (mode CPU)")
        GPU_AVAILABLE = False
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU          : non disponible (mode CPU)")
    GPU_AVAILABLE = False

# 2. PyTorch CUDA (si installe)
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    if cuda_ok:
        print(f"PyTorch CUDA : disponible ({torch.cuda.get_device_name(0)})")
        print(f"VRAM totale  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("PyTorch CUDA : non disponible")
except ImportError:
    print("PyTorch      : non installe (normal pour ce pipeline preprocessing)")

# 3. OpenCV GPU (optionnel)
try:
    import cv2
    build_info = cv2.getBuildInformation()
    cuda_in_cv2 = "CUDA" in build_info and "YES" in build_info[build_info.find("CUDA"):]
    print(f"OpenCV CUDA  : {'disponible' if cuda_in_cv2 else 'non disponible (CPU fallback OK)'}")
except ImportError:
    print("OpenCV       : non installe (sera utilise via pip install)")

print("=" * 60)
if GPU_AVAILABLE:
    print("OK - GPU pret. Le pipeline utilisera le GPU si disponible.")
else:
    print("INFO - Mode CPU. Le pipeline fonctionnera mais sera plus lent.")
    print("Pour activer le GPU : Settings > Accelerator > GPU T4 x2")
print("=" * 60)


---
## Module : `__init__.py`

Fichier d'initialisation du package Python `dq_agent`.
Il permet d'importer le package comme un module (`from dq_agent import ...`).


In [ ]:
%%writefile dq_agent/__init__.py"""Data & Quality Agent for BCMID — Multimodal Medical AI Pipeline."""__version__ = "1.0.0"__author__ = "Data & Quality Agent"

---
## 1. Module : `config.py` — Configuration centralisée

### Pourquoi ce module ?
En IA médicale, la **reproductibilité** est essentielle. Tous les paramètres du pipeline
(chemins des fichiers, taille cible des images, seuils de qualité, ratios de split)
sont centralisés dans un fichier `config.yaml` et chargés par ce module.

### Ce qu'il fait :
- Charge le fichier `config.yaml` et crée un objet `Config` avec des propriétés typées
- Définit les chemins dérivés automatiquement (`validated_dir`, `mammo_dir`, `us_dir`, `reports_dir`, `plots_dir`)
- Configure le logging (niveaux, format, fichier de log)
- Gère les valeurs par défaut pour tous les paramètres

### Choix techniques :
- **Dataclass Python** pour le typage fort et l'autocomplétion
- **YAML** (pas JSON) car plus lisible pour les humains et supporte les commentaires


In [ ]:
%%writefile dq_agent/config.py"""Configuration loader for the Data & Quality Agent."""from __future__ import annotationsimport osimport loggingfrom dataclasses import dataclass, fieldfrom pathlib import Pathfrom typing import Optionalimport yamlfrom dotenv import load_dotenvload_dotenv()  # charge .env si présent@dataclassclass PreprocessingConfig:    target_size: int = 512    mammo_clahe_clip_limit: float = 2.0    mammo_clahe_tile_grid: int = 8    us_median_ksize: int = 3    nlm_h: int = 5    bit_depth: int = 8@dataclassclass ModalityConfig:    require_report: bool = False    min_modalities: int = 2@dataclassclass IntegrityConfig:    min_dimension: int = 32    std_threshold: float = 1.0@dataclassclass SplitsConfig:    train_ratio: float = 0.70    val_ratio: float = 0.15    test_ratio: float = 0.15    random_seed: int = 42    rare_class_threshold: int = 5@dataclassclass LoggingConfig:    level: str = "INFO"    file: str = "dq_agent.log"@dataclassclass Config:    zip_path: Path = Path("")    tsv_path: Path = Path("")    work_dir: Path = Path("bcmid_work")    private_dir: Path = Path("bcmid_work")    preprocessing: PreprocessingConfig = field(default_factory=PreprocessingConfig)    modality_policy: ModalityConfig = field(default_factory=ModalityConfig)    integrity: IntegrityConfig = field(default_factory=IntegrityConfig)    splits: SplitsConfig = field(default_factory=SplitsConfig)    logging: LoggingConfig = field(default_factory=LoggingConfig)    @property    def raw_dir(self) -> Path:        return self.work_dir / "raw"    @property    def validated_dir(self) -> Path:        return self.work_dir / "validated"    @property    def mammo_dir(self) -> Path:        return self.validated_dir / "mammo"    @property    def us_dir(self) -> Path:        return self.validated_dir / "us"    @property    def reports_dir(self) -> Path:        return self.validated_dir / "reports"    @property    def plots_dir(self) -> Path:        return self.validated_dir / "plots"    @property    def mapping_file(self) -> Path:        """Private mapping — outside validated/."""        return self.private_dir / "id_mapping_PRIVATE.csv"    @property    def progress_file(self) -> Path:        return self.work_dir / ".progress.json"def load_config(path: str | Path = "config.yaml") -> Config:    """Load and validate configuration from a YAML file."""    path = Path(path)    if not path.exists():        raise FileNotFoundError(f"Config file not found: {path}")    with open(path, encoding="utf-8") as f:        raw = yaml.safe_load(f)    cfg = Config(        zip_path=Path(raw.get("zip_path", "")),        tsv_path=Path(raw.get("tsv_path", "")),        work_dir=Path(raw.get("work_dir", "bcmid_work")),        private_dir=Path(raw.get("private_dir", raw.get("work_dir", "bcmid_work"))),    )    if pp := raw.get("preprocessing"):        cfg.preprocessing = PreprocessingConfig(**pp)    if mp := raw.get("modality_policy"):        cfg.modality_policy = ModalityConfig(**mp)    if ig := raw.get("integrity"):        cfg.integrity = IntegrityConfig(**ig)    if sp := raw.get("splits"):        cfg.splits = SplitsConfig(**sp)    if lg := raw.get("logging"):        cfg.logging = LoggingConfig(**lg)    return cfgdef setup_logging(cfg: Config) -> None:    """Configure root logger with file + console handlers."""    level = getattr(logging, cfg.logging.level.upper(), logging.INFO)    fmt = "%(asctime)s [%(levelname)s] %(name)s — %(message)s"    handlers: list[logging.Handler] = [logging.StreamHandler()]    log_path = cfg.work_dir / cfg.logging.file    log_path.parent.mkdir(parents=True, exist_ok=True)    handlers.append(logging.FileHandler(log_path, encoding="utf-8"))    logging.basicConfig(level=level, format=fmt, handlers=handlers, force=True)

---
## 2. Module : `zip_io.py` — Lecture du ZIP sans extraction

### Pourquoi ce module ?
Le fichier BCMID.zip fait **12,9 Go**. L'extraire entièrement doublerait l'espace disque nécessaire.
Ce module permet de **scanner la structure** et **lire les images à la volée** sans extraction préalable.

### Ce qu'il fait :
- `scan_zip()` : parcourt l'archive ZIP et construit un dictionnaire `{patient_id: {us: [...], mammo: [...], report: ...}}`
- `read_image_from_zip()` : lit une image directement depuis le ZIP en mémoire (PIL)
- Détecte automatiquement les sous-dossiers (`Ultrasound/`, `Mammogram/`) de façon **tolérante à la casse**

### Choix techniques :
- Utilise `zipfile.ZipFile` en mode lecture seule (pas d'extraction)
- Compatible avec les noms de dossiers variés (`US`, `ultrasound`, `Mammogram`, `mammo`...)


In [ ]:
%%writefile dq_agent/zip_io.py"""ZIP I/O utilities — dual-mode: ZIP file OR extracted directory (Kaggle-compatible)."""from __future__ import annotationsimport ioimport loggingimport zipfilefrom pathlib import Path, PurePosixPathfrom typing import Generatorfrom PIL import Imagelogger = logging.getLogger(__name__)# Folder name aliases for modalities_US_ALIASES = {"ultrasound", "us", "echo", "echography", "ultrason"}_MAMMO_ALIASES = {"mammogram", "mammography", "mammo", "mg", "mammographie"}def _is_us_folder(name: str) -> bool:    return name.lower() in _US_ALIASESdef _is_mammo_folder(name: str) -> bool:    return name.lower() in _MAMMO_ALIASESdef _is_image(fname: str) -> bool:    return fname.lower().endswith((".png", ".jpg", ".jpeg", ".dcm", ".bmp", ".tiff", ".tif"))def _is_pdf(fname: str) -> bool:    return fname.lower().endswith(".pdf")# ══════════════════════════════════════════════════════════════════# AUTO-DETECTION (like notebook1)# ══════════════════════════════════════════════════════════════════def auto_find_dataset(search_root: Path = Path("/kaggle/input")) -> tuple[Path | None, Path | None]:    """    Auto-detect BCMID dataset like notebook1 does.    Recursively searches for BCMID_annotations.tsv and derives data root.    Returns (data_root, tsv_path) or (None, None) if not found.    """    # Search in kaggle input first, then cwd    for root in [search_root, Path(".")]:        if not root.exists():            continue        candidates = list(root.rglob("BCMID_annotations.tsv"))        if candidates:            tsv_path = candidates[0]            data_root = tsv_path.parent            logger.info("Auto-detected TSV: %s", tsv_path)            logger.info("Auto-detected data root: %s", data_root)            return data_root, tsv_path    return None, Nonedef auto_find_zip(search_root: Path = Path("/kaggle/input")) -> Path | None:    """Find a BCMID ZIP file under search_root."""    for root in [search_root, Path(".")]:        if not root.exists():            continue        for candidate in root.rglob("*.zip"):            name = candidate.name.lower()            if "bcmid" in name or "breast" in name:                return candidate    return None# ══════════════════════════════════════════════════════════════════# DIRECTORY MODE (Kaggle extracted dataset)# ══════════════════════════════════════════════════════════════════def scan_dir(data_root: Path) -> dict[str, dict]:    """    Scan an extracted directory structure (Kaggle layout).    Expected structure:        data_root/          [BCMID/]            <patient_id>/              Mammogram/                *.png              Ultrasound/                *.png              Report.pdf  (optional)    Returns dict mapping patient_id -> {        "us": [Path, ...],        "mammo": [Path, ...],        "report": [Path, ...],        "source": "dir",        "patient_dir": Path,    }    """    patients: dict[str, dict] = {}    for directory in sorted(data_root.rglob("*")):        if not directory.is_dir():            continue        child_names = {c.name.lower() for c in directory.iterdir() if c.is_dir()}        if not any(_is_mammo_folder(n) or _is_us_folder(n) for n in child_names):            continue        pid = directory.name        entry: dict = {            "us": [],            "mammo": [],            "report": [],            "other": [],            "source": "dir",            "patient_dir": directory,        }        for child in directory.iterdir():            if child.is_dir():                if _is_us_folder(child.name):                    entry["us"] = sorted([f for f in child.iterdir() if f.is_file() and _is_image(f.name)])                elif _is_mammo_folder(child.name):                    entry["mammo"] = sorted([f for f in child.iterdir() if f.is_file() and _is_image(f.name)])            elif child.is_file():                if _is_pdf(child.name):                    entry["report"].append(child)        patients[pid] = entry    logger.info("scan_dir: found %d patient directories under %s", len(patients), data_root)    return patients# ══════════════════════════════════════════════════════════════════# ZIP MODE (local or Kaggle with uploaded ZIP)# ══════════════════════════════════════════════════════════════════def scan_zip(zip_path: Path) -> dict[str, dict]:    """    Scan zip structure and return per-patient inventory.    Returns dict mapping patient_id -> {        "us": [zip_member_str, ...],        "mammo": [zip_member_str, ...],        "report": [zip_member_str, ...],        "source": "zip",    }    """    patients: dict[str, dict] = {}    with zipfile.ZipFile(zip_path, "r") as zf:        for info in zf.infolist():            parts = PurePosixPath(info.filename).parts            if len(parts) < 1 or not parts[0]:                continue            pid = parts[0]            if pid not in patients:                patients[pid] = {"us": [], "mammo": [], "report": [], "other": [], "source": "zip"}            if len(parts) == 1:                continue            fname = parts[-1]            if not fname:                continue            if len(parts) == 3:                sub = parts[1]                if _is_us_folder(sub) and _is_image(fname):                    patients[pid]["us"].append(info.filename)                elif _is_mammo_folder(sub) and _is_image(fname):                    patients[pid]["mammo"].append(info.filename)                else:                    patients[pid]["other"].append(info.filename)            elif len(parts) == 2:                if _is_pdf(fname):                    patients[pid]["report"].append(info.filename)                elif _is_image(fname):                    patients[pid]["other"].append(info.filename)    logger.info("scan_zip: found %d patient directories in %s", len(patients), zip_path)    return patients# ══════════════════════════════════════════════════════════════════# UNIFIED SCAN — auto-detects ZIP vs directory# ══════════════════════════════════════════════════════════════════def scan_source(zip_path: Path | None = None, data_root: Path | None = None) -> dict[str, dict]:    """    Unified scan: uses ZIP if available, otherwise extracted directory.    Priority: zip_path > data_root    """    if zip_path and Path(zip_path).is_file():        logger.info("Source mode: ZIP")        return scan_zip(Path(zip_path))    elif data_root and Path(data_root).is_dir():        logger.info("Source mode: directory")        return scan_dir(Path(data_root))    else:        raise FileNotFoundError(            f"No valid data source. zip_path={zip_path}, data_root={data_root}"        )# ══════════════════════════════════════════════════════════════════# IMAGE READING — unified for ZIP and directory# ══════════════════════════════════════════════════════════════════def read_image_from_zip(zf: zipfile.ZipFile, member: str) -> Image.Image | None:    """Read a single image from an open ZipFile."""    try:        data = zf.read(member)    except Exception as exc:        logger.warning("Cannot read zip member %s: %s", member, exc)        return None    if member.lower().endswith(".dcm"):        try:            import pydicom            import numpy as np            ds = pydicom.dcmread(io.BytesIO(data))            arr = ds.pixel_array.astype(float)            if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":                arr = arr.max() - arr            arr = (arr - arr.min()) / (arr.ptp() + 1e-8) * 255            return Image.fromarray(arr.astype("uint8"), mode="L")        except Exception as exc:            logger.warning("DICOM read error for %s: %s", member, exc)            return None    else:        try:            img = Image.open(io.BytesIO(data))            img.load()            return img        except Exception as exc:            logger.warning("Image read error for %s: %s", member, exc)            return Nonedef read_image_from_path(path: Path) -> Image.Image | None:    """Read a single image from a filesystem path."""    try:        img = Image.open(path)        img.load()        return img    except Exception as exc:        logger.warning("Image read error for %s: %s", path, exc)        return Nonedef read_image(source: str | Path, zf: zipfile.ZipFile | None = None) -> Image.Image | None:    """Unified image reader: ZIP member (string + zf) or filesystem path."""    if zf is not None and isinstance(source, str):        return read_image_from_zip(zf, source)    return read_image_from_path(Path(source))# ══════════════════════════════════════════════════════════════════# PATIENT ITERATOR# ══════════════════════════════════════════════════════════════════def iter_patients(    zip_path: Path | None,    patients: dict[str, dict],    limit: int | None = None,) -> Generator[tuple[str, dict, zipfile.ZipFile | None], None, None]:    """    Yield (patient_id, inventory, open_zipfile_or_None) for each patient.    Automatically handles ZIP mode (zf open) or directory mode (zf=None).    """    pids = list(patients.keys())    if limit is not None:        pids = pids[:limit]    # Detect mode from inventory    source_mode = patients[pids[0]].get("source", "zip") if pids else "zip"    if source_mode == "zip" and zip_path and Path(zip_path).is_file():        with zipfile.ZipFile(zip_path, "r") as zf:            for pid in pids:                yield pid, patients[pid], zf    else:        for pid in pids:            yield pid, patients[pid], None

---
## 3. Module : `profiling.py` — Statistiques du dataset

### Pourquoi ce module ?
Avant tout traitement, il faut **comprendre les données**. Ce module produit un profil complet
du dataset pour détecter les anomalies et orienter les choix de prétraitement.

### Ce qu'il fait :
- Compte le nombre de patients dans le ZIP vs dans le TSV (détecte l'écart annoncé 332 vs 323)
- Calcule les statistiques par modalité (nb images/patient, tailles min/max/moy)
- Identifie les extensions de fichiers et les modes d'image (L, RGBA...)
- Construit la distribution des classes BI-RADS
- Détecte les patients sans rapport PDF

### Résultats clés trouvés :
- **332 patients** dans le ZIP et **332 dans le TSV** → écart réel = 0 (le brief annonçait 323, c'était faux)
- **2001 images** PNG uniquement (pas de DICOM, JPG ou TIFF)
- **11 patients** sans rapport PDF


In [ ]:
%%writefile dq_agent/profiling.py"""Module 1 — Profiling: dataset statistics and BI-RADS distribution."""from __future__ import annotationsimport jsonimport loggingimport reimport zipfilefrom pathlib import Pathfrom typing import Anyimport numpy as npimport pandas as pdfrom tqdm import tqdmfrom .config import Configfrom .zip_io import read_image_from_zip, scan_ziplogger = logging.getLogger(__name__)# Pattern: {digits}y{suffix} — suffix encodes study year (e.g. "19" = 2019)_ID_PATTERN = re.compile(r"^(\d+)y(\d{2,3})$")def _parse_age_from_id(pid: str) -> int | None:    """Extract year suffix from patient ID (not the real age — just the year code)."""    m = _ID_PATTERN.match(pid)    if m:        return int(m.group(2))    return Nonedef profile_dataset(    cfg: Config,    patients: dict[str, dict],    labels: dict[str, str],    n_total_zip_patients: int | None = None,) -> dict[str, Any]:    """    Compute dataset-level profiling statistics.    Parameters    ----------    cfg : Config    patients : output of zip_io.scan_zip() (may be limited)    labels : dict mapping patient_id -> BI-RADS label (from validation step)    n_total_zip_patients : total patient dirs in ZIP (before --limit). If None, uses len(patients).    Returns    -------    dict with profiling stats    """    zip_path = cfg.zip_path    n_zip = n_total_zip_patients if n_total_zip_patients is not None else len(patients)    n_tsv = len(set(labels.values()) - {"UNKNOWN"})  # patients actually in TSV    # n_patients_in_annotations is set by validation and passed through labels    # But labels dict includes patients with UNKNOWN label (not in TSV)    # So: patients with real labels = patients where label != UNKNOWN    n_annotated = sum(1 for v in labels.values() if v != "UNKNOWN")    stats: dict[str, Any] = {        "n_patient_folders": n_zip,        "n_patients_in_annotations": n_annotated,        "ecart_patients": abs(n_zip - n_annotated),        "patients_only_in_folders": sorted(set(patients) - set(labels)),        "patients_only_in_annotations": sorted(set(labels) - set(patients)),        "image_extensions": {},        "modality_counts": {"us": 0, "mammo": 0},        "per_patient": [],        "birads_distribution": {},        "year_suffix_distribution": {},        "images_without_report": 0,    }    ext_counter: dict[str, int] = {}    birads_counter: dict[str, int] = {}    year_counter: dict[str, int] = {}    us_counts: list[int] = []    mammo_counts: list[int] = []    logger.info("Profiling %d patients …", len(patients))    with zipfile.ZipFile(zip_path, "r") as zf:        for pid, inv in tqdm(patients.items(), desc="Profiling", unit="patient"):            n_us = len(inv["us"])            n_mammo = len(inv["mammo"])            has_report = len(inv["report"]) > 0            label = labels.get(pid, "UNKNOWN")            us_counts.append(n_us)            mammo_counts.append(n_mammo)            if not has_report:                stats["images_without_report"] += 1            # Count extensions            for member in inv["us"] + inv["mammo"]:                ext = Path(member).suffix.lower().lstrip(".")                ext_counter[ext] = ext_counter.get(ext, 0) + 1            # Sample one US + one mammo image for intensity stats            us_intensity: float | None = None            mammo_intensity: float | None = None            if inv["us"]:                img = read_image_from_zip(zf, inv["us"][0])                if img is not None:                    arr = np.array(img.convert("L"), dtype=float)                    us_intensity = float(arr.mean())            if inv["mammo"]:                img = read_image_from_zip(zf, inv["mammo"][0])                if img is not None:                    arr = np.array(img.convert("L"), dtype=float)                    mammo_intensity = float(arr.mean())            # Year suffix            yr = _parse_age_from_id(pid)            if yr is not None:                year_counter[str(yr)] = year_counter.get(str(yr), 0) + 1            # BI-RADS            birads_counter[label] = birads_counter.get(label, 0) + 1            stats["per_patient"].append(                {                    "patient_id": pid,  # will be pseudonymised later                    "n_us": n_us,                    "n_mammo": n_mammo,                    "has_report": has_report,                    "birads": label,                    "year_suffix": yr,                    "us_mean_intensity": us_intensity,                    "mammo_mean_intensity": mammo_intensity,                }            )    stats["image_extensions"] = ext_counter    stats["modality_counts"]["us"] = sum(us_counts)    stats["modality_counts"]["mammo"] = sum(mammo_counts)    stats["birads_distribution"] = dict(sorted(birads_counter.items()))    stats["year_suffix_distribution"] = dict(sorted(year_counter.items(), key=lambda x: int(x[0])))    if us_counts:        stats["us_per_patient"] = {            "min": int(min(us_counts)),            "max": int(max(us_counts)),            "mean": round(float(np.mean(us_counts)), 2),            "std": round(float(np.std(us_counts)), 2),        }    if mammo_counts:        stats["mammo_per_patient"] = {            "min": int(min(mammo_counts)),            "max": int(max(mammo_counts)),            "mean": round(float(np.mean(mammo_counts)), 2),            "std": round(float(np.std(mammo_counts)), 2),        }    logger.info(        "Profiling done — %d patients, BI-RADS classes: %s",        len(patients),        list(stats["birads_distribution"].keys()),    )    return statsdef save_profile(profile: dict[str, Any], out_dir: Path) -> None:    """Persist profiling data (without real IDs in the public version)."""    out_dir.mkdir(parents=True, exist_ok=True)    # Save public version (no patient IDs)    public = {k: v for k, v in profile.items() if k != "per_patient"}    with open(out_dir / "profile_summary.json", "w", encoding="utf-8") as f:        json.dump(public, f, indent=2, ensure_ascii=False)    logger.info("Profile summary saved to %s", out_dir / "profile_summary.json")

---
## 4. Module : `integrity.py` — Vérification d'intégrité

### Pourquoi ce module ?
Un fichier image peut exister sur le disque mais être **corrompu, vide, ou trop petit** pour être utile.
Ce module scanne chaque image et exclut celles qui poseraient problème à l'entraînement du modèle.

### Ce qu'il fait :
- **Lisibilité** : tente d'ouvrir chaque image avec PIL. Si ça échoue → marquée corrompue
- **Uniformité** : calcule l'écart-type des pixels. Si quasi-nul (image toute noire/blanche) → exclue
- **Dimensions minimales** : rejette les images trop petites (< seuil configurable)
- **Doublons** : calcule un hash MD5 de chaque fichier et regroupe les images identiques

### Résultats clés trouvés :
- **1 image corrompue** : `0165y09/Ultrasound/0165y09_US_1.png` (impossible à décoder)
- **6 groupes de doublons** : tous intra-patient (échographies dupliquées au sein du même dossier)
- **0 image uniforme**


In [ ]:
%%writefile dq_agent/integrity.py"""Module 2 — Integrity: corruption detection, duplicates, uniform images.Supports both ZIP mode and directory mode (Kaggle extracted dataset)."""from __future__ import annotationsimport hashlibimport ioimport loggingimport zipfilefrom pathlib import Pathfrom typing import Anyimport numpy as npfrom PIL import Imagefrom tqdm import tqdmfrom .config import Configfrom .zip_io import read_image, read_image_from_ziplogger = logging.getLogger(__name__)def _hash_bytes(data: bytes) -> str:    return hashlib.sha256(data).hexdigest()def _is_uniform(arr: np.ndarray, threshold: float) -> bool:    return float(arr.std()) < thresholddef _read_raw_bytes(source: str | Path, zf: zipfile.ZipFile | None) -> bytes | None:    """Read raw bytes from ZIP member or filesystem path."""    if zf is not None and isinstance(source, str):        try:            return zf.read(source)        except Exception as exc:            logger.warning("Cannot read zip member %s: %s", source, exc)            return None    else:        try:            return Path(source).read_bytes()        except Exception as exc:            logger.warning("Cannot read file %s: %s", source, exc)            return Nonedef check_integrity(    cfg: Config,    patients: dict[str, dict],) -> dict[str, Any]:    """    Check integrity of all images — works for ZIP and directory sources.    Returns    -------    dict with:      - corrupted_or_invalid: list of {member, reason}      - uniform_images: list of members      - duplicate_files: list of lists (groups of identical files by hash)      - valid_members: set of members that passed all checks    """    min_dim = cfg.integrity.min_dimension    std_thresh = cfg.integrity.std_threshold    corrupted: list[dict] = []    uniform: list[str] = []    hash_map: dict[str, list[str]] = {}    valid_members: set[str] = set()    # Collect all image references    all_images: list[tuple[str | Path, str | Path]] = []  # (source_ref, display_name)    for inv in patients.values():        for m in inv.get("us", []) + inv.get("mammo", []):            display = str(m)            all_images.append((m, display))    logger.info("Checking integrity of %d images ...", len(all_images))    # Determine if we're in ZIP or directory mode    first_inv = next(iter(patients.values()), {})    source_mode = first_inv.get("source", "zip")    if source_mode == "zip":        zip_path = cfg.zip_path        if not Path(zip_path).is_file():            raise FileNotFoundError(f"ZIP not found: {zip_path}")        _check_zip_mode(all_images, zip_path, corrupted, uniform, hash_map, valid_members, min_dim, std_thresh)    else:        _check_dir_mode(all_images, corrupted, uniform, hash_map, valid_members, min_dim, std_thresh)    duplicates = [members for members in hash_map.values() if len(members) > 1]    logger.info(        "Integrity done — valid=%d, corrupted=%d, uniform=%d, duplicate_groups=%d",        len(valid_members), len(corrupted), len(uniform), len(duplicates),    )    return {        "corrupted_or_invalid": corrupted,        "corrupted_count": len(corrupted),        "corrupted_sample": corrupted[:20],        "uniform_images": uniform,        "duplicate_files": duplicates,        "duplicate_count": len(duplicates),        "valid_members": valid_members,    }def _check_zip_mode(all_images, zip_path, corrupted, uniform, hash_map, valid_members, min_dim, std_thresh):    with zipfile.ZipFile(zip_path, "r") as zf:        for source, display in tqdm(all_images, desc="Integrity", unit="img"):            member = str(source)            data = _read_raw_bytes(member, zf)            if data is None:                corrupted.append({"member": display, "reason": "unreadable"})                continue            h = _hash_bytes(data)            hash_map.setdefault(h, []).append(display)            img = read_image_from_zip(zf, member)            if img is None:                corrupted.append({"member": display, "reason": "cannot decode image"})                continue            w, h_px = img.size            if w < min_dim or h_px < min_dim:                corrupted.append({"member": display, "reason": f"too small: {w}x{h_px}"})                continue            arr = np.array(img.convert("L"), dtype=float)            if float(arr.std()) < std_thresh:                uniform.append(display)                corrupted.append({"member": display, "reason": "uniform (std < threshold)"})                continue            valid_members.add(display)def _check_dir_mode(all_images, corrupted, uniform, hash_map, valid_members, min_dim, std_thresh):    for source, display in tqdm(all_images, desc="Integrity", unit="img"):        path = Path(source)        data = _read_raw_bytes(path, None)        if data is None:            corrupted.append({"member": display, "reason": "unreadable"})            continue        h = _hash_bytes(data)        hash_map.setdefault(h, []).append(display)        try:            img = Image.open(io.BytesIO(data))            img.load()        except Exception as exc:            corrupted.append({"member": display, "reason": f"cannot decode image: {exc}"})            continue        w, h_px = img.size        if w < min_dim or h_px < min_dim:            corrupted.append({"member": display, "reason": f"too small: {w}x{h_px}"})            continue        arr = np.array(img.convert("L"), dtype=float)        if float(arr.std()) < std_thresh:            uniform.append(display)            corrupted.append({"member": display, "reason": "uniform (std < threshold)"})            continue        valid_members.add(display)

---
## 5. Module : `modality.py` — Classification des modalités

### Pourquoi ce module ?
Pour un entraînement **multimodal** (mammo + écho), on a besoin de savoir quels patients
ont les deux modalités et lesquels n'en ont qu'une.

### Ce qu'il fait :
- Classe chaque patient en : `complete` (mammo + écho), `mammo_seule`, `echo_seule`, ou `vide`
- Le statut se base **uniquement sur les images** (pas sur la présence du rapport PDF)
- Sépare les patients multimodaux (pour le train multimodal) des partiels (pour un éventuel train unimodal)

### Résultats clés :
- **332/332 patients** sont `complete` (tous ont les deux modalités)


In [ ]:
%%writefile dq_agent/modality.py"""Module 3 — Modality classification: complete / mammo_only / us_only / empty."""from __future__ import annotationsimport loggingfrom typing import Anyfrom .config import Configlogger = logging.getLogger(__name__)# Canonical status valuesSTATUS_COMPLETE = "complete"STATUS_MAMMO_ONLY = "mammo_only"STATUS_US_ONLY = "us_only"STATUS_EMPTY = "empty"STATUS_PARTIAL = "partial"  # has both modalities but missing reportdef classify_patients(    cfg: Config,    patients: dict[str, dict],    valid_members: set[str],) -> dict[str, dict]:    """    Classify each patient by modality completeness.    Parameters    ----------    cfg : Config    patients : zip inventory from scan_zip()    valid_members : set of zip members that passed integrity checks    Returns    -------    dict mapping patient_id -> {        "status": str,        "n_valid_us": int,        "n_valid_mammo": int,        "has_report": bool,        "valid_us_members": list,        "valid_mammo_members": list,    }    """    result: dict[str, dict] = {}    require_report = cfg.modality_policy.require_report    for pid, inv in patients.items():        valid_us = [m for m in inv["us"] if m in valid_members]        valid_mammo = [m for m in inv["mammo"] if m in valid_members]        has_report = len(inv["report"]) > 0        has_us = len(valid_us) > 0        has_mammo = len(valid_mammo) > 0        # Status based ONLY on images — report presence is metadata, not a gate        if has_us and has_mammo:            status = STATUS_COMPLETE        elif has_mammo:            status = STATUS_MAMMO_ONLY        elif has_us:            status = STATUS_US_ONLY        else:            status = STATUS_EMPTY        result[pid] = {            "status": status,            "n_valid_us": len(valid_us),            "n_valid_mammo": len(valid_mammo),            "has_report": has_report,            "valid_us_members": valid_us,            "valid_mammo_members": valid_mammo,        }    counts = {}    for v in result.values():        s = v["status"]        counts[s] = counts.get(s, 0) + 1    logger.info("Modality classification — %s", counts)    return resultdef modality_summary(modalities: dict[str, dict]) -> dict[str, int]:    """Return status counts."""    counts: dict[str, int] = {}    for v in modalities.values():        s = v["status"]        counts[s] = counts.get(s, 0) + 1    return countsdef filter_for_training(    modalities: dict[str, dict],) -> tuple[list[str], list[str]]:    """    Split patients into multimodal (for training) and partial (for unimodal use).    Returns    -------    (multimodal_ids, partial_ids)    """    multimodal = [pid for pid, v in modalities.items() if v["status"] == STATUS_COMPLETE]    partial = [pid for pid, v in modalities.items() if v["status"] != STATUS_COMPLETE]    logger.info(        "Training split — multimodal=%d, partial/excluded=%d",        len(multimodal),        len(partial),    )    return multimodal, partial

---
## 6. Module : `validation.py` — Validation des labels BI-RADS

### Pourquoi ce module ?
Les annotations d'un dataset médical peuvent contenir des erreurs : labels manquants, mal formés,
patients dans le TSV mais pas dans les images, etc. Ce module fait la **jointure** entre
le fichier d'annotations (TSV) et les dossiers patients du ZIP.

### Ce qu'il fait :
- **Détection automatique des colonnes** du TSV (pas de noms hardcodés)
- **Normalisation des labels** : `'4A'`, `'4a'`, `'BI-RADS 4A'`, `'BIRADS 4A'` → tous convertis en `'4A'`
- Détecte : patients sans label, labels sans dossier, IDs dupliqués, labels hors plage
- Capture les colonnes supplémentaires (`Binary Label`, `Pathology Label`)

### Résultats clés :
- Colonnes détectées : `Patient ID`, `BI-RADS Label`, `Binary Label`, `Pathology Label`
- **0 anomalie** : tous les patients ont un label, tous les labels ont un dossier


In [ ]:
%%writefile dq_agent/validation.py"""Module 6 — Validation: join images with BI-RADS labels, detect anomalies."""from __future__ import annotationsimport loggingimport refrom pathlib import Pathfrom typing import Anyimport pandas as pdfrom .config import Configlogger = logging.getLogger(__name__)# BI-RADS normalization patterns_BIRADS_PATTERNS = [    # Match "BI-RADS 4A", "BIRADS4a", "bi rads 4A" etc.    re.compile(r"bi[\s\-_]*rads[\s\-_]*(\d[A-Ca-c]?)", re.IGNORECASE),    # Match bare numbers/letters like "4A", "4a", "4", "0"    re.compile(r"^(\d[A-Ca-c]?)$", re.IGNORECASE),]_VALID_BIRADS = {    "0", "1", "2", "3",    "4", "4A", "4B", "4C",    "5", "6",}def normalize_birads(raw: str) -> str:    """    Normalize a raw BI-RADS label string to canonical form.    Examples    --------    "4A" -> "4A"    "4a" -> "4A"    "BI-RADS 4A" -> "4A"    "bi rads 4b" -> "4B"    "4" -> "4"    """    raw = str(raw).strip()    for pat in _BIRADS_PATTERNS:        m = pat.search(raw)        if m:            val = m.group(1).upper()            return val    return raw.upper().strip()def load_annotations(tsv_path: Path) -> tuple[pd.DataFrame, list[str]]:    """    Load and inspect the annotations TSV.    Returns    -------    (df, column_names)    df has columns auto-detected; patient_id_col and birads_col identified heuristically.    """    if not tsv_path.exists():        raise FileNotFoundError(f"Annotations TSV not found: {tsv_path}")    df = pd.read_csv(tsv_path, sep="\t", dtype=str)    df.columns = [c.strip() for c in df.columns]    logger.info("TSV loaded — columns: %s, rows: %d", list(df.columns), len(df))    return df, list(df.columns)def detect_id_and_label_columns(df: pd.DataFrame) -> tuple[str, str]:    """    Heuristically identify the patient ID column and the BI-RADS label column.    Works with the real BCMID TSV columns: 'Patient ID', 'BI-RADS Label', etc.    """    cols = [c.lower() for c in df.columns]    orig_cols = list(df.columns)    id_col = None    label_col = None    # ID column heuristics — covers "Patient ID", "patient_id", "id", etc.    for kw in ["patient id", "patient_id", "patientid", "id", "patient", "case", "subject"]:        for i, c in enumerate(cols):            if kw in c and "label" not in c:                id_col = orig_cols[i]                break        if id_col:            break    # Label column heuristics — covers "BI-RADS Label", "birads", "label", etc.    for kw in ["bi-rads", "birads", "bi_rads", "bi rads"]:        for i, c in enumerate(cols):            if kw in c:                label_col = orig_cols[i]                break        if label_col:            break    # Fallback to any column named "label" or "class"    if label_col is None:        for kw in ["label", "class", "category", "grade"]:            for i, c in enumerate(cols):                if kw in c and i != cols.index(id_col.lower() if id_col else ""):                    label_col = orig_cols[i]                    break            if label_col:                break    # Last-resort fallback    if id_col is None:        id_col = orig_cols[0]        logger.warning("Could not detect ID column — using first column: %s", id_col)    if label_col is None:        label_col = orig_cols[-1]        logger.warning("Could not detect label column — using last column: %s", label_col)    logger.info("Detected columns — ID: '%s', Label: '%s'", id_col, label_col)    return id_col, label_coldef validate_labels(    cfg: Config,    patients: dict[str, dict],) -> dict[str, Any]:    """    Full validation of BI-RADS labels against patient folders.    Returns    -------    dict with:      - labels: dict patient_id -> normalized_birads      - issues: list of issue dicts      - duplicate_ids_in_annotations: list      - patients_without_label: list      - labels_without_folder: list      - malformed_labels: list    """    tsv_path = cfg.tsv_path    issues: list[dict] = []    # Handle missing TSV gracefully    if not tsv_path.exists():        logger.warning("TSV not found at %s — labels will be UNKNOWN", tsv_path)        labels = {pid: "UNKNOWN" for pid in patients}        return {            "labels": labels,            "issues": [{"type": "missing_tsv", "detail": str(tsv_path)}],            "duplicate_ids_in_annotations": [],            "patients_without_label": list(patients.keys()),            "labels_without_folder": [],            "malformed_labels": [],            "tsv_columns": [],            "id_column": None,            "label_column": None,        }    df, cols = load_annotations(tsv_path)    id_col, label_col = detect_id_and_label_columns(df)    # Capture any extra informative columns    extra_cols = [c for c in cols if c not in (id_col, label_col)]    extra_data: dict[str, dict] = {}    # Detect duplicate IDs in TSV    dup_ids = df[df.duplicated(subset=[id_col], keep=False)][id_col].tolist()    if dup_ids:        issues.append({"type": "duplicate_ids_in_annotations", "ids": list(set(dup_ids))})        logger.warning("Duplicate IDs in annotations: %s", dup_ids[:10])    # Deduplicate (keep first)    df = df.drop_duplicates(subset=[id_col], keep="first")    # Store extra column data per patient    for _, row in df.iterrows():        pid = str(row[id_col]).strip()        extra_data[pid] = {c: str(row[c]).strip() for c in extra_cols}    # Normalize labels    raw_labels = df.set_index(id_col)[label_col].to_dict()    normalized_labels: dict[str, str] = {}    malformed: list[dict] = []    for pid_raw, lbl in raw_labels.items():        pid = str(pid_raw).strip()        norm = normalize_birads(str(lbl))        if norm not in _VALID_BIRADS:            malformed.append({"patient_id": pid, "raw_label": lbl, "normalized": norm})            issues.append({"type": "malformed_label", "patient_id": pid, "raw": lbl, "normalized": norm})        normalized_labels[pid] = norm    # Cross-join    folder_ids = set(patients.keys())    annotation_ids = set(normalized_labels.keys())    patients_without_label = sorted(folder_ids - annotation_ids)    labels_without_folder = sorted(annotation_ids - folder_ids)    for pid in patients_without_label:        issues.append({"type": "patient_without_label", "patient_id": pid})        normalized_labels[pid] = "UNKNOWN"    if labels_without_folder:        issues.append({"type": "labels_without_folder", "ids": labels_without_folder})    logger.info(        "Validation — %d labels loaded, %d without label, %d labels without folder, %d malformed",        len(raw_labels),        len(patients_without_label),        len(labels_without_folder),        len(malformed),    )    return {        "labels": normalized_labels,        "issues": issues,        "duplicate_ids_in_annotations": list(set(dup_ids)),        "patients_without_label": patients_without_label,        "labels_without_folder": labels_without_folder,        "malformed_labels": malformed,        "tsv_columns": cols,        "id_column": id_col,        "label_column": label_col,        "extra_columns": extra_cols if 'extra_cols' in dir() else [],        "extra_data": extra_data if 'extra_data' in dir() else {},        "n_patients_in_annotations": len(df) if 'df' in dir() else 0,    }

---
## 7. Module : `preprocess.py` — Prétraitement des images médicales

### Pourquoi ce module ?
Les images brutes ne sont pas directement utilisables par un réseau de neurones.
Chaque modalité d'imagerie médicale a ses propres artefacts de bruit qu'il faut traiter différemment.

### Pipeline de prétraitement :

**Pour l'échographie (US) :**
1. Conversion en niveaux de gris
2. **Filtre médian** (taille 5×5) → réduit le *speckle* (bruit granuleux typique des ultrasons)
3. Normalisation [0, 1]
4. Redimensionnement à 512×512 (interpolation INTER_AREA pour le downscaling)
5. Sauvegarde en PNG 8 bits

**Pour la mammographie :**
1. Conversion en niveaux de gris
2. **Non-Local Means (NLM)** → débruitage avancé qui préserve les bords et les micro-calcifications
3. **CLAHE** (Contrast Limited Adaptive Histogram Equalization) → rehausse le contraste local des tissus
4. Normalisation [0, 1]
5. Redimensionnement à 512×512
6. Sauvegarde en PNG 8 bits

### Choix de 512×512 (vs 224×224 du notebook1) :
En imagerie médicale, les détails fins (micro-calcifications, contours de masses) sont critiques.
224×224 perd trop d'information. 512×512 est le meilleur compromis résolution/mémoire GPU.

### Idempotence :
Si le pipeline est relancé, les images déjà traitées sont **ignorées** (vérification par fichier existant).


In [ ]:
%%writefile dq_agent/preprocess.py"""Module 4 — Preprocessing: normalization, denoising, CLAHE, resize.Supports both ZIP mode and directory mode (Kaggle extracted dataset)."""from __future__ import annotationsimport loggingimport zipfilefrom pathlib import Pathfrom typing import Anyimport numpy as npfrom PIL import Imagefrom tqdm import tqdmfrom .config import Configfrom .zip_io import read_imagelogger = logging.getLogger(__name__)def _to_gray_array(img: Image.Image) -> np.ndarray:    """Convert PIL image to grayscale float32 array [0, 1]."""    gray = img.convert("L")    return np.array(gray, dtype=np.float32) / 255.0def _normalize_01(arr: np.ndarray) -> np.ndarray:    mn, mx = arr.min(), arr.max()    if mx - mn < 1e-8:        return np.zeros_like(arr)    return (arr - mn) / (mx - mn)def _median_denoise(arr: np.ndarray, ksize: int) -> np.ndarray:    """Apply median filter (for ultrasound speckle noise)."""    try:        import cv2  # type: ignore        u8 = (arr * 255).clip(0, 255).astype(np.uint8)        filtered = cv2.medianBlur(u8, ksize)        return filtered.astype(np.float32) / 255.0    except ImportError:        from scipy.ndimage import median_filter  # type: ignore        return median_filter(arr, size=ksize)def _nlm_denoise(arr: np.ndarray, h: int) -> np.ndarray:    """Apply Non-Local Means denoising (for mammography)."""    try:        import cv2  # type: ignore        u8 = (arr * 255).clip(0, 255).astype(np.uint8)        denoised = cv2.fastNlMeansDenoising(u8, h=h, templateWindowSize=7, searchWindowSize=21)        return denoised.astype(np.float32) / 255.0    except ImportError:        logger.warning("cv2 not available, skipping NLM denoising")        return arrdef _apply_clahe(arr: np.ndarray, clip_limit: float, tile_grid: int) -> np.ndarray:    """Apply CLAHE to enhance local contrast (for mammography)."""    try:        import cv2  # type: ignore        u8 = (arr * 255).clip(0, 255).astype(np.uint8)        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid, tile_grid))        enhanced = clahe.apply(u8)        return enhanced.astype(np.float32) / 255.0    except ImportError:        logger.warning("cv2 not available, skipping CLAHE")        return arrdef _resize(arr: np.ndarray, target: int) -> np.ndarray:    """Resize to target×target using INTER_AREA (best for downscaling)."""    try:        import cv2  # type: ignore        u8 = (arr * 255).clip(0, 255).astype(np.uint8)        return cv2.resize(u8, (target, target), interpolation=cv2.INTER_AREA)    except ImportError:        pil = Image.fromarray((arr * 255).astype(np.uint8))        pil = pil.resize((target, target), Image.LANCZOS)        return np.array(pil)def preprocess_us(arr: np.ndarray, cfg: Config) -> np.ndarray:    """Full preprocessing pipeline for ultrasound images."""    arr = _normalize_01(arr)    arr = _median_denoise(arr, cfg.preprocessing.us_median_ksize)    arr = _normalize_01(arr)    return _resize(arr, cfg.preprocessing.target_size)def preprocess_mammo(arr: np.ndarray, cfg: Config) -> np.ndarray:    """Full preprocessing pipeline for mammography images."""    arr = _normalize_01(arr)    arr = _nlm_denoise(arr, cfg.preprocessing.nlm_h)    arr = _apply_clahe(arr, cfg.preprocessing.mammo_clahe_clip_limit, cfg.preprocessing.mammo_clahe_tile_grid)    arr = _normalize_01(arr)    return _resize(arr, cfg.preprocessing.target_size)def process_patient(    pid: str,    pseudo_id: str,    modality_info: dict,    cfg: Config,    zf: zipfile.ZipFile | None,) -> dict[str, Any]:    """    Preprocess all images for one patient and save to validated/.    Works for both ZIP (zf provided) and directory (zf=None) modes.    Skips already-processed images (idempotent).    """    mammo_dir = cfg.mammo_dir    us_dir = cfg.us_dir    mammo_dir.mkdir(parents=True, exist_ok=True)    us_dir.mkdir(parents=True, exist_ok=True)    saved_us: list[str] = []    saved_mammo: list[str] = []    # --- Ultrasound ---    for idx, member in enumerate(modality_info.get("valid_us_members", [])):        out_name = f"{pseudo_id}_us_{idx}.png"        out_path = us_dir / out_name        if out_path.exists():            saved_us.append(str(out_path.relative_to(cfg.validated_dir)))            continue        img = read_image(member, zf)        if img is None:            logger.warning("Skip US image (unreadable): %s", member)            continue        arr = _to_gray_array(img)        processed = preprocess_us(arr, cfg)        Image.fromarray(processed, mode="L").save(out_path, format="PNG")        saved_us.append(str(out_path.relative_to(cfg.validated_dir)))    # --- Mammography ---    for idx, member in enumerate(modality_info.get("valid_mammo_members", [])):        out_name = f"{pseudo_id}_mammo_{idx}.png"        out_path = mammo_dir / out_name        if out_path.exists():            saved_mammo.append(str(out_path.relative_to(cfg.validated_dir)))            continue        img = read_image(member, zf)        if img is None:            logger.warning("Skip mammo image (unreadable): %s", member)            continue        arr = _to_gray_array(img)        processed = preprocess_mammo(arr, cfg)        Image.fromarray(processed, mode="L").save(out_path, format="PNG")        saved_mammo.append(str(out_path.relative_to(cfg.validated_dir)))    return {        "pseudo_id": pseudo_id,        "us_paths": saved_us,        "mammo_paths": saved_mammo,        "n_us": len(saved_us),        "n_mammo": len(saved_mammo),    }

---
## 8. Module : `anonymize.py` — Pseudonymisation (RGPD/HIPAA)

### Pourquoi ce module ?
En médecine, les identifiants patients sont des **données sensibles**. Le RGPD (Europe)
et HIPAA (USA) imposent leur protection. Ce module remplace chaque ID réel par un pseudonyme
**irréversible** sans la clé secrète.

### Algorithme :
```
pseudo_id = 'P' + SHA256(sel_secret + id_réel)[:10]
```
- Le **sel** est lu depuis la variable d'environnement `DQ_SALT` (jamais dans le code !)
- Le résultat est **déterministe** : même sel + même ID → même pseudo à chaque exécution
- La table de correspondance `id_mapping_PRIVATE.csv` est sauvegardée **hors du dossier partageable**
- Elle est ajoutée au `.gitignore`

### Garantie :**
- Aucun ID réel dans `validated/` (noms de fichiers, manifest, rapports texte)
- Vérifié automatiquement par une recherche regex dans tous les fichiers de sortie


In [ ]:
%%writefile dq_agent/anonymize.py"""Module 5 — Anonymization: SHA-256 pseudonymisation with environment salt."""from __future__ import annotationsimport csvimport hashlibimport loggingimport osfrom pathlib import Pathfrom dotenv import load_dotenvload_dotenv()logger = logging.getLogger(__name__)_SALT_ENV_VAR = "DQ_SALT"_PSEUDO_PREFIX = "P"_HASH_CHARS = 10def _get_salt() -> str:    """Read salt from environment variable or raise."""    salt = os.environ.get(_SALT_ENV_VAR, "")    if not salt:        # Use a deterministic but non-empty default for development only        logger.warning(            "Environment variable %s not set — using default development salt. "            "Set %s in your .env file for production!",            _SALT_ENV_VAR,            _SALT_ENV_VAR,        )        salt = "bcmid_dev_salt_change_me"    return saltdef pseudonymise(patient_id: str, salt: str | None = None) -> str:    """    Generate a pseudonym for a patient ID.    Format: P + first 10 hex chars of SHA-256(salt + patient_id)    Deterministic: same inputs always produce the same output.    """    if salt is None:        salt = _get_salt()    h = hashlib.sha256((salt + patient_id).encode("utf-8")).hexdigest()    return _PSEUDO_PREFIX + h[:_HASH_CHARS]def build_mapping(patient_ids: list[str], salt: str | None = None) -> dict[str, str]:    """    Build a dict mapping real_id -> pseudo_id for all patients.    Parameters    ----------    patient_ids : list of real patient folder names    salt : override salt (for testing)    """    if salt is None:        salt = _get_salt()    mapping: dict[str, str] = {}    seen_pseudos: set[str] = set()    for pid in patient_ids:        pseudo = pseudonymise(pid, salt)        if pseudo in seen_pseudos:            # Collision (astronomically rare) — extend hash            h = hashlib.sha256((salt + pid + "extra").encode()).hexdigest()            pseudo = _PSEUDO_PREFIX + h[:_HASH_CHARS]        seen_pseudos.add(pseudo)        mapping[pid] = pseudo    return mappingdef save_private_mapping(mapping: dict[str, str], out_path: Path) -> None:    """    Save the real_id -> pseudo_id mapping to a PRIVATE CSV file.    This file must NEVER be shared or committed to git.    """    out_path.parent.mkdir(parents=True, exist_ok=True)    with open(out_path, "w", newline="", encoding="utf-8") as f:        writer = csv.writer(f)        writer.writerow(["real_id", "pseudo_id"])        for real_id, pseudo_id in sorted(mapping.items()):            writer.writerow([real_id, pseudo_id])    logger.info("Private mapping saved to %s (DO NOT SHARE)", out_path)def load_private_mapping(path: Path) -> dict[str, str]:    """Load real_id -> pseudo_id mapping from CSV."""    mapping: dict[str, str] = {}    with open(path, newline="", encoding="utf-8") as f:        reader = csv.DictReader(f)        for row in reader:            mapping[row["real_id"]] = row["pseudo_id"]    return mapping

---
## 9. Module : `reports.py` — Extraction et sécurisation des rapports PDF

### Pourquoi ce module ?
Le dataset BCMID contient un **rapport médical PDF** par patient. Pour un pipeline multimodal
(images + texte), il faut extraire ce texte. Mais attention : les rapports contiennent
des **informations personnelles** (noms, dates) et souvent **le label BI-RADS** lui-même.

### Ce qu'il fait :
1. **Extraction du texte** via `pdfplumber` (gère les PDF natifs et les tableaux)
2. **Nettoyage PII** : remplace les noms, dates, numéros de téléphone, IDs patients par des balises
   (`<NOM>`, `<DATE>`, `<PHONE>`, `<ID>`)
3. **Détection du label leakage** : regex cherchant les mentions de `BI-RADS`, `BIRADS`, `bi-rads`...
4. Sauvegarde en `reports/<pseudo_id>.txt`

### Résultat critique découvert :
**96% des rapports mentionnent explicitement le score BI-RADS !**
→ Si l'Agent Vision reçoit le texte brut, il pourra "tricher" en lisant la réponse au lieu
d'analyser l'image. C'est un risque de **label leakage** majeur, signalé dans le rapport de qualité.


In [ ]:
%%writefile dq_agent/reports.py"""Module reports — PDF text extraction, PII removal, label leakage detection.For each patient:  1. Extract text from PDF report via pdfplumber.  2. Strip identifying information (names, dates, numeric IDs, real patient ID).  3. Save as reports/<pseudo_id>.txt in validated/.  4. Detect whether the text mentions BI-RADS (label leakage risk for Vision agent)."""from __future__ import annotationsimport ioimport loggingimport reimport zipfilefrom pathlib import Pathfrom typing import Anylogger = logging.getLogger(__name__)# ── PII removal patterns ──────────────────────────────────────────────────────_PII_PATTERNS: list[tuple[str, str]] = [    # Dates: DD/MM/YYYY, DD-MM-YYYY, MM/DD/YYYY, YYYY-MM-DD, "le 12 janvier 2021"    (r"\b\d{1,2}[/\-\.]\d{1,2}[/\-\.]\d{2,4}\b", "<DATE>"),    (r"\b\d{4}[/\-\.]\d{1,2}[/\-\.]\d{1,2}\b", "<DATE>"),    (r"\b\d{1,2}\s+(janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)\s+\d{4}\b",     "<DATE>", ),    # Phone numbers    (r"\b(?:\+?\d[\s\-]?){8,15}\b", "<PHONE>"),    # Emails    (r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b", "<EMAIL>"),    # Social security / ID numbers (long digit sequences 6-20 chars)    (r"\b\d{6,20}\b", "<ID_NUM>"),    # Names heuristic: two or more consecutive Capitalized words (Title Case)    (r"\b([A-ZÁÉÈÊÀÙÛÔÎÂ][a-záéèêàùûôîâ]+)(\s+[A-ZÁÉÈÊÀÙÛÔÎÂ][a-záéèêàùûôîâ]+){1,3}\b",     "<NOM>"),]# ── BI-RADS mention detection ─────────────────────────────────────────────────_BIRADS_MENTION = re.compile(    r"bi[\s\-]*rads|birads|acr\s*\d|cat[eé]gorie\s*\d|classification\s*\d",    re.IGNORECASE,)def _strip_pii(text: str, real_id: str) -> str:    """Remove PII from extracted text. Also removes the real patient ID."""    # Remove real patient ID first (most specific)    text = re.sub(re.escape(real_id), "<PATIENT_ID>", text, flags=re.IGNORECASE)    # Remove year suffix pattern (NNNNyNN)    text = re.sub(r"\b\d+y\d+\b", "<PATIENT_ID>", text)    for pattern_args in _PII_PATTERNS:        if len(pattern_args) == 3:            pattern, replacement, *_ = pattern_args        else:            pattern, replacement = pattern_args        try:            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE | re.UNICODE)        except re.error:            pass    return textdef extract_pdf_text(pdf_bytes: bytes) -> tuple[str, str]:    """    Extract text from PDF bytes using pdfplumber.    Returns    -------    (text, status)  where status is 'ok' | 'scanned' | 'empty' | 'error'    """    try:        import pdfplumber  # type: ignore        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:            pages_text = []            for page in pdf.pages:                t = page.extract_text()                if t:                    pages_text.append(t)            full_text = "\n".join(pages_text).strip()    except Exception as exc:        logger.warning("pdfplumber error: %s", exc)        return "", "error"    if not full_text:        return "", "scanned"   # PDF with no extractable text (image-based)    return full_text, "ok"def has_birads_mention(text: str) -> bool:    """Return True if text contains a BI-RADS classification mention."""    return bool(_BIRADS_MENTION.search(text))def process_reports(    patients: dict[str, dict],    mapping: dict[str, str],    zip_path: Path,    reports_dir: Path,) -> dict[str, Any]:    """    Extract, anonymize, and save all patient PDF reports.    Parameters    ----------    patients : zip inventory from scan_zip()    mapping  : real_id -> pseudo_id    zip_path : path to the BCMID ZIP    reports_dir : output directory for .txt files    Returns    -------    dict with summary stats for quality_report    """    from tqdm import tqdm    reports_dir.mkdir(parents=True, exist_ok=True)    n_ok = 0    n_scanned = 0    n_error = 0    n_no_pdf = 0    n_birads_mention = 0    n_total = 0    with zipfile.ZipFile(zip_path, "r") as zf:        for pid, inv in tqdm(patients.items(), desc="Reports", unit="patient"):            pseudo = mapping.get(pid, pid)            out_path = reports_dir / f"{pseudo}.txt"            if not inv["report"]:                n_no_pdf += 1                # Write a placeholder                out_path.write_text("[NO_REPORT_AVAILABLE]\n", encoding="utf-8")                continue            # Idempotent            if out_path.exists():                text = out_path.read_text(encoding="utf-8")                if has_birads_mention(text):                    n_birads_mention += 1                n_total += 1                n_ok += 1                continue            member = inv["report"][0]            try:                pdf_bytes = zf.read(member)            except Exception as exc:                logger.warning("Cannot read PDF %s: %s", member, exc)                n_error += 1                out_path.write_text("[PDF_READ_ERROR]\n", encoding="utf-8")                continue            text, status = extract_pdf_text(pdf_bytes)            n_total += 1            if status == "ok":                n_ok += 1                # Strip PII                clean_text = _strip_pii(text, real_id=pid)                if has_birads_mention(clean_text):                    n_birads_mention += 1                out_path.write_text(clean_text, encoding="utf-8")            elif status == "scanned":                n_scanned += 1                out_path.write_text("[SCANNED_PDF_NO_TEXT]\n", encoding="utf-8")            else:                n_error += 1                out_path.write_text("[PDF_ERROR]\n", encoding="utf-8")    leakage_pct = round(n_birads_mention / n_total * 100, 1) if n_total > 0 else 0.0    logger.info(        "Reports — ok=%d, scanned=%d, error=%d, no_pdf=%d | BI-RADS mention: %d/%d (%.1f%%)",        n_ok, n_scanned, n_error, n_no_pdf, n_birads_mention, n_total, leakage_pct,    )    return {        "reports_extracted_ok": n_ok,        "reports_scanned_unreadable": n_scanned,        "reports_error": n_error,        "reports_no_pdf": n_no_pdf,        "reports_total_with_pdf": n_total,        "birads_mention_in_reports": n_birads_mention,        "label_leakage_risk_pct": leakage_pct,    }

---
## 10. Module : `splits.py` — Découpage train/val/test

### Pourquoi ce module ?
Un mauvais découpage du dataset peut **fausser complètement** les résultats d'un modèle d'IA.

### Règles critiques respectées :
1. **Split par patient** (jamais par image) : empêche qu'une image du même patient soit
   dans le train ET dans le test (data leakage)
2. **Stratification par `label_group`** : chaque split respecte la proportion des classes BI-RADS
3. **Graine fixe** (`seed=42`) : résultats 100% reproductibles
4. **Pas de fusion automatique de BI-RADS 0 ou 1 avec 2** (directive clinique)

### Groupement proposé :
| BI-RADS | `label_group` | Patients |
|---|---|---|
| 0 | `incomplete` | 5 |
| 1, 2 | `benign` | 217 |
| 3 | `probably_benign` | 47 |
| 4 | `suspicious` | 24 |
| 5 | `highly_suspicious` | 15 |
| 6 | `known_malignant` | 24 |

### Split final : train=228 / val=47 / test=57


In [ ]:
%%writefile dq_agent/splits.py"""Module 8 — Splits: stratified train/val/test split by patient (no data leakage).label_group is added as metadata but splits are stratified on it (not on raw birads).The grouping must be validated by the user before calling make_splits."""from __future__ import annotationsimport jsonimport loggingimport mathimport randomfrom collections import Counter, defaultdictfrom pathlib import Pathfrom typing import Anyfrom .config import Configlogger = logging.getLogger(__name__)# ── Proposed clinical grouping (to be validated by user) ─────────────────────DEFAULT_LABEL_GROUPS: dict[str, str] = {    "0": "incomplete",    "1": "benign",    "2": "benign",    "3": "probably_benign",    "4": "suspicious",    "4A": "suspicious",    "4B": "suspicious",    "4C": "suspicious",    "5": "highly_suspicious",    "6": "known_malignant",    "UNKNOWN": "unknown",    "RARE": "rare",}def get_label_group(birads: str, groups: dict[str, str] | None = None) -> str:    """Map a normalized BI-RADS label to its clinical group."""    g = groups or DEFAULT_LABEL_GROUPS    return g.get(birads, "unknown")def show_distribution(patient_labels: dict[str, str]) -> None:    """Log BI-RADS distribution and proposed groups."""    counts = Counter(patient_labels.values())    logger.info("BI-RADS distribution (raw):")    for cls in sorted(counts):        grp = get_label_group(cls)        logger.info("  BI-RADS %s → group '%s' : %d patients", cls, grp, counts[cls])def _group_rare_classes(    patient_labels: dict[str, str],    threshold: int,    label_groups: dict[str, str],) -> tuple[dict[str, str], dict[str, str]]:    """    Group BI-RADS classes with fewer than threshold patients into 'RARE'.    Classes 0 and 1 are NEVER merged with 2 automatically.    Returns    -------    (strat_labels, rare_mapping)    strat_labels: patient_id -> label_group (used for stratification)    rare_mapping: original_class -> 'rare' (documented)    """    group_counts = Counter(        get_label_group(lbl, label_groups) for lbl in patient_labels.values()    )    rare_groups = {grp for grp, cnt in group_counts.items() if cnt < threshold}    if rare_groups:        logger.info(            "Groups with < %d patients (rare): %s — will be pooled as 'rare' for splits only",            threshold, sorted(rare_groups),        )    strat_labels: dict[str, str] = {}    rare_mapping: dict[str, str] = {}    for pid, birads in patient_labels.items():        grp = get_label_group(birads, label_groups)        if grp in rare_groups:            strat_labels[pid] = "rare"            rare_mapping[birads] = "rare"        else:            strat_labels[pid] = grp    return strat_labels, rare_mappingdef make_splits(    cfg: Config,    multimodal_ids: list[str],    labels: dict[str, str],    label_groups: dict[str, str] | None = None,) -> dict[str, Any]:    """    Create stratified train/val/test splits by patient.    Stratification is by label_group. BI-RADS 0 and 1 are NEVER auto-merged with 2.    No patient appears in more than one split.    Returns    -------    dict with train/val/test lists of patient IDs and metadata.    """    rng = random.Random(cfg.splits.random_seed)    threshold = cfg.splits.rare_class_threshold    lg = label_groups or DEFAULT_LABEL_GROUPS    # Separate known vs unknown labels    patient_labels = {        pid: labels.get(pid, "UNKNOWN")        for pid in multimodal_ids        if labels.get(pid, "UNKNOWN") != "UNKNOWN"    }    unknown_ids = [pid for pid in multimodal_ids if labels.get(pid, "UNKNOWN") == "UNKNOWN"]    if unknown_ids:        logger.warning(            "%d patients with UNKNOWN label — added to train set.", len(unknown_ids)        )    show_distribution(patient_labels)    # Group by clinical category for stratification    strat_labels, rare_mapping = _group_rare_classes(patient_labels, threshold, lg)    # Stratify buckets    class_buckets: dict[str, list[str]] = defaultdict(list)    for pid, grp in strat_labels.items():        class_buckets[grp].append(pid)    for grp in class_buckets:        rng.shuffle(class_buckets[grp])    train_ids: list[str] = []    val_ids: list[str] = []    test_ids: list[str] = []    train_r = cfg.splits.train_ratio    val_r = cfg.splits.val_ratio    for grp, pids in class_buckets.items():        n = len(pids)        n_train = max(1, math.floor(n * train_r))        n_val = max(0, math.floor(n * val_r))        n_val = min(n_val, n - n_train)        n_test = n - n_train - n_val        train_ids.extend(pids[:n_train])        val_ids.extend(pids[n_train: n_train + n_val])        test_ids.extend(pids[n_train + n_val:])    train_ids.extend(unknown_ids)    rng.shuffle(train_ids)    rng.shuffle(val_ids)    rng.shuffle(test_ids)    # ── No-leakage invariant ──    assert set(train_ids).isdisjoint(set(val_ids)),  "LEAK: patient(s) in train AND val"    assert set(train_ids).isdisjoint(set(test_ids)), "LEAK: patient(s) in train AND test"    assert set(val_ids).isdisjoint(set(test_ids)),   "LEAK: patient(s) in val AND test"    logger.info(        "Splits — train=%d, val=%d, test=%d (seed=%d)",        len(train_ids), len(val_ids), len(test_ids), cfg.splits.random_seed,    )    return {        "train": train_ids,        "val": val_ids,        "test": test_ids,        "n_train": len(train_ids),        "n_val": len(val_ids),        "n_test": len(test_ids),        "random_seed": cfg.splits.random_seed,        "rare_groups_pooled": rare_mapping,        "rare_threshold": threshold,        "unknown_label_patients_in_train": unknown_ids,        "label_groups_used": lg,        "class_distribution": {            "train": dict(Counter(strat_labels.get(p, "unknown") for p in train_ids)),            "val":   dict(Counter(strat_labels.get(p, "unknown") for p in val_ids)),            "test":  dict(Counter(strat_labels.get(p, "unknown") for p in test_ids)),        },    }def save_splits(splits: dict[str, Any], out_dir: Path) -> None:    """Save splits.json."""    out_dir.mkdir(parents=True, exist_ok=True)    path = out_dir / "splits.json"    with open(path, "w", encoding="utf-8") as f:        json.dump(splits, f, indent=2, ensure_ascii=False)    logger.info("Splits saved: %s", path)

---
## 11. Module : `report.py` — Rapport de qualité

### Ce qu'il génère :
- `quality_report.json` : rapport structuré lisible par une machine (pour l'orchestrateur n8n)
- `quality_report.md` : rapport lisible par un humain avec tableaux Markdown

### Contenu du rapport :
- Nombre de patients (ZIP vs TSV), écart détecté
- Distribution BI-RADS
- Statuts de modalité (complet/partiel)
- Images corrompues/invalides (nombre + échantillon des 20 premières)
- Groupes de doublons
- Patients sans label, labels sans dossier, IDs dupliqués
- Statistiques des rapports PDF (extraits, scannés, leakage %)
- Paramètres de prétraitement utilisés
- Versions des librairies


In [ ]:
%%writefile dq_agent/report.py"""Module 7 — Report: generate quality_report.json and quality_report.md."""from __future__ import annotationsimport importlib.metadataimport jsonimport loggingfrom datetime import datetime, timezonefrom pathlib import Pathfrom typing import Anylogger = logging.getLogger(__name__)_LIBRARY_VERSIONS = ["Pillow", "numpy", "pandas", "scikit-learn", "tqdm", "pyyaml", "pydicom"]def _get_versions() -> dict[str, str]:    versions: dict[str, str] = {}    for lib in _LIBRARY_VERSIONS:        try:            versions[lib] = importlib.metadata.version(lib)        except Exception:            versions[lib] = "N/A"    return versionsdef build_report(    profile: dict[str, Any],    integrity_result: dict[str, Any],    modality_counts: dict[str, int],    validation_result: dict[str, Any],    preprocessing_params: dict[str, Any],    n_multimodal_ready: int,    reports_stats: dict[str, Any] | None = None,) -> dict[str, Any]:    """Assemble the quality_report.json payload."""    report: dict[str, Any] = {        "date_generated": datetime.now(timezone.utc).isoformat(),        "library_versions": _get_versions(),        # Dataset counts        "n_patient_folders": profile["n_patient_folders"],        "n_patients_in_annotations": validation_result.get("n_patients_in_annotations", 0),        "ecart_patients": profile["ecart_patients"],        "patients_only_in_folders": profile["patients_only_in_folders"],        "patients_only_in_annotations": profile["patients_only_in_annotations"],        # Modality        "modality_status": modality_counts,        "patients_ready_multimodal": n_multimodal_ready,        # BI-RADS        "birads_distribution": profile["birads_distribution"],        # Integrity        "corrupted_or_invalid_count": integrity_result["corrupted_count"],        "corrupted_or_invalid_sample": integrity_result["corrupted_sample"],        "duplicate_files_count": integrity_result["duplicate_count"],        "duplicate_groups": integrity_result["duplicate_files"],        # Annotations quality        "patients_without_label": validation_result["patients_without_label"],        "labels_without_folder": validation_result["labels_without_folder"],        "duplicate_ids_in_annotations": validation_result["duplicate_ids_in_annotations"],        "malformed_labels": validation_result["malformed_labels"],        "tsv_columns_detected": validation_result.get("tsv_columns", []),        "id_column_detected": validation_result.get("id_column"),        "label_column_detected": validation_result.get("label_column"),        "extra_tsv_columns": validation_result.get("extra_columns", []),        # Dataset stats        "image_extensions": profile["image_extensions"],        "us_per_patient": profile.get("us_per_patient", {}),        "mammo_per_patient": profile.get("mammo_per_patient", {}),        "patients_without_report_pdf": profile.get("images_without_report", 0),        # Reports extraction        "reports_stats": reports_stats or {},        "label_leakage_risk_pct": (reports_stats or {}).get("label_leakage_risk_pct", None),        # Preprocessing        "parametres_preprocessing": preprocessing_params,        # Validation issues        "all_issues": validation_result.get("issues", []),    }    return reportdef save_report(report: dict[str, Any], out_dir: Path) -> None:    """Save JSON and Markdown reports."""    out_dir.mkdir(parents=True, exist_ok=True)    json_path = out_dir / "quality_report.json"    with open(json_path, "w", encoding="utf-8") as f:        json.dump(report, f, indent=2, ensure_ascii=False)    logger.info("JSON report saved: %s", json_path)    md_path = out_dir / "quality_report.md"    _write_markdown(report, md_path)    logger.info("Markdown report saved: %s", md_path)def _fmt_list(lst: list, max_items: int = 10) -> str:    if not lst:        return "_aucun_"    shown = [str(x) for x in lst[:max_items]]    rest = len(lst) - max_items    result = ", ".join(shown)    if rest > 0:        result += f" _… et {rest} autres_"    return resultdef _write_markdown(report: dict[str, Any], path: Path) -> None:    lines: list[str] = []    def h(level: int, text: str) -> None:        lines.append("\n" + "#" * level + " " + text + "\n")    def row(*cells: Any) -> str:        return "| " + " | ".join(str(c) for c in cells) + " |"    h(1, "Rapport de Qualité — Data & Quality Agent (BCMID)")    lines.append(f"**Généré le :** {report['date_generated']}\n")    h(2, "1. Résumé Dataset")    lines.append(row("Métrique", "Valeur"))    lines.append(row("---", "---"))    lines.append(row("Dossiers patients (ZIP)", report["n_patient_folders"]))    lines.append(row("Patients dans les annotations (TSV)", report["n_patients_in_annotations"]))    lines.append(row("**Écart détecté**", f"**{report['ecart_patients']}**"))    lines.append(row("Patients prêts (multimodal complet)", report["patients_ready_multimodal"]))    lines.append(row("Patients sans rapport PDF", report["patients_without_report_pdf"]))    h(2, "2. Distribution BI-RADS")    lines.append(row("Classe BI-RADS", "Nombre de patients"))    lines.append(row("---", "---"))    for k, v in report["birads_distribution"].items():        lines.append(row(k, v))    h(2, "3. Statuts de Modalité")    lines.append(row("Statut", "Nombre"))    lines.append(row("---", "---"))    for k, v in report["modality_status"].items():        lines.append(row(k, v))    h(2, "4. Intégrité des Images")    lines.append(row("Indicateur", "Valeur"))    lines.append(row("---", "---"))    lines.append(row("Images corrompues / invalides", report["corrupted_or_invalid_count"]))    lines.append(row("Groupes de doublons exacts", report["duplicate_files_count"]))    if report["corrupted_or_invalid_sample"]:        h(3, "Exemples d'images invalides (max 20)")        lines.append(row("Fichier", "Raison"))        lines.append(row("---", "---"))        for item in report["corrupted_or_invalid_sample"]:            lines.append(row(item.get("member", "?"), item.get("reason", "?")))    h(2, "5. Annotations BI-RADS")    lines.append(row("Indicateur", "Valeur"))    lines.append(row("---", "---"))    lines.append(row("Colonnes TSV détectées", ", ".join(report["tsv_columns_detected"])))    lines.append(row("Colonne ID détectée", report["id_column_detected"] or "N/A"))    lines.append(row("Colonne label détectée", report["label_column_detected"] or "N/A"))    lines.append(row("Patients sans label", len(report["patients_without_label"])))    lines.append(row("Labels sans dossier", len(report["labels_without_folder"])))    lines.append(row("IDs dupliqués dans TSV", len(report["duplicate_ids_in_annotations"])))    lines.append(row("Labels mal formés", len(report["malformed_labels"])))    if report["patients_without_label"]:        lines.append(f"\n**Patients sans label :** {_fmt_list(report['patients_without_label'])}\n")    if report["labels_without_folder"]:        lines.append(f"\n**Labels sans dossier :** {_fmt_list(report['labels_without_folder'])}\n")    h(2, "6. Statistiques Images")    for modality, key in [("Échographie (US)", "us_per_patient"), ("Mammographie", "mammo_per_patient")]:        stats = report.get(key, {})        if stats:            lines.append(f"\n**{modality}** — min={stats.get('min')}, max={stats.get('max')}, "                         f"moy={stats.get('mean')}, écart-type={stats.get('std')}\n")    h(2, "7. Paramètres de Prétraitement")    lines.append(row("Paramètre", "Valeur"))    lines.append(row("---", "---"))    for k, v in report["parametres_preprocessing"].items():        lines.append(row(k, v))    h(2, "8. Versions des Librairies")    lines.append(row("Librairie", "Version"))    lines.append(row("---", "---"))    for lib, ver in report["library_versions"].items():        lines.append(row(lib, ver))    path.write_text("\n".join(lines), encoding="utf-8")

---
## 12. Module : `plots.py` — Visualisations

### 6 graphiques générés :
1. **Distribution BI-RADS** : barplot du nombre de patients par classe
2. **Images par patient** : histogramme du nombre d'images (US + mammo) par patient
3. **Répartition des modalités** : camembert complet/mammo_seule/echo_seule
4. **Histogramme des tailles** : distribution des résolutions d'images originales
5. **Avant/après mammographie** : comparaison visuelle du prétraitement (CLAHE + NLM)
6. **Avant/après échographie** : comparaison visuelle (filtre médian anti-speckle)


In [ ]:
%%writefile dq_agent/plots.py"""Profiling plots: BI-RADS distribution, images per patient, modality breakdown, before/after."""from __future__ import annotationsimport loggingimport zipfilefrom pathlib import Pathfrom typing import Anyimport matplotlibmatplotlib.use("Agg")  # headlessimport matplotlib.pyplot as pltimport numpy as npfrom .config import Configfrom .zip_io import read_image_from_ziplogger = logging.getLogger(__name__)def _save(fig: plt.Figure, path: Path) -> None:    path.parent.mkdir(parents=True, exist_ok=True)    fig.savefig(path, dpi=120, bbox_inches="tight")    plt.close(fig)    logger.info("Plot saved: %s", path)def plot_birads_distribution(birads: dict[str, int], out_dir: Path) -> None:    """Bar chart of BI-RADS class distribution."""    fig, ax = plt.subplots(figsize=(8, 5))    classes = list(birads.keys())    counts = list(birads.values())    colors = plt.cm.tab10(range(len(classes)))    ax.bar(classes, counts, color=colors)    ax.set_title("Distribution des classes BI-RADS")    ax.set_xlabel("Classe BI-RADS")    ax.set_ylabel("Nombre de patients")    for i, v in enumerate(counts):        ax.text(i, v + 0.3, str(v), ha="center", fontsize=9)    ax.set_ylim(0, max(counts) * 1.15 if counts else 1)    _save(fig, out_dir / "birads_distribution.png")def plot_images_per_patient(profile: dict[str, Any], out_dir: Path) -> None:    """Histograms of US and mammo image counts per patient."""    per_patient = profile.get("per_patient", [])    if not per_patient:        return    us_counts = [p["n_us"] for p in per_patient]    mammo_counts = [p["n_mammo"] for p in per_patient]    fig, axes = plt.subplots(1, 2, figsize=(12, 5))    for ax, data, title in [        (axes[0], us_counts, "Images échographie (US) par patient"),        (axes[1], mammo_counts, "Images mammographie par patient"),    ]:        bins = range(0, max(data) + 2) if data else [0, 1]        ax.hist(data, bins=bins, color="#4C72B0", edgecolor="white", rwidth=0.85)        ax.set_title(title)        ax.set_xlabel("Nombre d'images")        ax.set_ylabel("Nombre de patients")        ax.set_xticks(list(bins))    fig.suptitle("Nombre d'images par patient", fontsize=13)    _save(fig, out_dir / "images_per_patient.png")def plot_modality_distribution(modality_counts: dict[str, int], out_dir: Path) -> None:    """Pie chart of modality status."""    labels = list(modality_counts.keys())    sizes = list(modality_counts.values())    if not any(sizes):        return    fig, ax = plt.subplots(figsize=(7, 7))    ax.pie(        sizes,        labels=labels,        autopct="%1.1f%%",        startangle=140,        colors=plt.cm.Set2(range(len(labels))),    )    ax.set_title("Répartition des statuts de modalité")    _save(fig, out_dir / "modality_distribution.png")def plot_image_size_histogram(profile: dict[str, Any], zip_path: Path, out_dir: Path) -> None:    """Sample image dimensions histogram (from first 50 patients)."""    per_patient = profile.get("per_patient", [])    widths, heights = [], []    sample_members: list[str] = []    for pp in per_patient[:50]:        pid = pp["patient_id"]        # We can't easily get members back from profile alone; skip silently        pass    # Quick scan from zip    try:        from .zip_io import scan_zip        with zipfile.ZipFile(zip_path, "r") as zf:            names = zf.namelist()            sample = [n for n in names if n.lower().endswith(".png")][:200]            for member in sample:                try:                    from PIL import Image                    import io                    data = zf.read(member)                    img = Image.open(io.BytesIO(data))                    widths.append(img.width)                    heights.append(img.height)                except Exception:                    pass    except Exception as exc:        logger.warning("Could not sample image sizes: %s", exc)        return    if not widths:        return    fig, axes = plt.subplots(1, 2, figsize=(12, 5))    axes[0].hist(widths, bins=30, color="#DD8452", edgecolor="white")    axes[0].set_title("Distribution largeur (pixels)")    axes[0].set_xlabel("Largeur")    axes[0].set_ylabel("Nombre d'images")    axes[1].hist(heights, bins=30, color="#55A868", edgecolor="white")    axes[1].set_title("Distribution hauteur (pixels)")    axes[1].set_xlabel("Hauteur")    fig.suptitle("Tailles des images originales (échantillon 200)", fontsize=13)    _save(fig, out_dir / "image_size_histogram.png")def plot_before_after(    cfg: Config,    patients: dict[str, dict],    modalities: dict[str, dict],    out_dir: Path,) -> None:    """    Show before/after preprocessing for 1 mammo + 1 US patient.    Reads directly from ZIP without saving full preprocessed image.    """    from .preprocess import _to_gray_array, preprocess_mammo, preprocess_us    zip_path = cfg.zip_path    # Pick first complete patients    mammo_pid = us_pid = None    for pid, info in modalities.items():        if info["valid_mammo_members"] and mammo_pid is None:            mammo_pid = pid        if info["valid_us_members"] and us_pid is None:            us_pid = pid        if mammo_pid and us_pid:            break    with zipfile.ZipFile(zip_path, "r") as zf:        for pid, member_key, title, preprocess_fn in [            (mammo_pid, "valid_mammo_members", "Mammographie", preprocess_mammo),            (us_pid, "valid_us_members", "Échographie", preprocess_us),        ]:            if pid is None:                continue            members = modalities[pid][member_key]            if not members:                continue            img = read_image_from_zip(zf, members[0])            if img is None:                continue            arr_before = _to_gray_array(img)            arr_after = preprocess_fn(arr_before.copy(), cfg)            fig, axes = plt.subplots(1, 2, figsize=(10, 5))            axes[0].imshow(arr_before, cmap="gray")            axes[0].set_title(f"{title} — Original")            axes[0].axis("off")            axes[1].imshow(arr_after, cmap="gray")            axes[1].set_title(f"{title} — Après prétraitement (512×512)")            axes[1].axis("off")            fig.suptitle(f"Avant / Après — {title}", fontsize=13)            fname = "before_after_mammo.png" if "mammo" in member_key else "before_after_us.png"            _save(fig, out_dir / fname)def generate_all_plots(    cfg: Config,    profile: dict[str, Any],    modality_counts: dict[str, int],    patients: dict[str, dict],    modalities: dict[str, dict],) -> None:    """Generate and save all profiling plots."""    out_dir = cfg.plots_dir    out_dir.mkdir(parents=True, exist_ok=True)    plot_birads_distribution(profile["birads_distribution"], out_dir)    plot_images_per_patient(profile, out_dir)    plot_modality_distribution(modality_counts, out_dir)    plot_image_size_histogram(profile, cfg.zip_path, out_dir)    plot_before_after(cfg, patients, modalities, out_dir)    logger.info("All plots saved to %s", out_dir)

---
## 13. Module : `cli.py` — Orchestration du pipeline

### Ce qu'il fait :
C'est le **chef d'orchestre** qui appelle tous les modules dans le bon ordre :

```
1. Scan ZIP → 2. Validation labels → 3. Intégrité → 4. Modalités
→ 5. Profiling → 6. Anonymisation → 7. Prétraitement images
→ 8. Extraction PDF → 9. Manifestes → 10. Splits → 11. Plots → 12. Rapport
```

### Fonctionnalités clés :
- `--limit N` : traiter seulement N patients (pour les tests rapides)
- **Idempotence** : fichier `.progress.json` pour ne pas retraiter les patients déjà faits
- **Robustesse** : un fichier corrompu ne fait jamais planter le pipeline
- Sous-commandes : `run`, `profile`, `splits`, `report`


In [ ]:
%%writefile dq_agent/cli.py"""CLI entry point for the Data & Quality Agent.Usage-----    python -m dq_agent run    --config config.yaml [--limit N]    python -m dq_agent profile --config config.yaml [--limit N]    python -m dq_agent splits  --config config.yaml    python -m dq_agent report  --config config.yaml"""from __future__ import annotationsimport argparseimport csvimport jsonimport loggingimport sysimport zipfilefrom pathlib import Pathfrom typing import Anyfrom .config import Config, load_config, setup_loggingfrom .splits import DEFAULT_LABEL_GROUPS, get_label_grouplogger = logging.getLogger(__name__)# ──────────────────────────── helpers ────────────────────────────────────────def _load_progress(cfg: Config) -> dict[str, Any]:    if cfg.progress_file.exists():        with open(cfg.progress_file, encoding="utf-8") as f:            return json.load(f)    return {}def _save_progress(cfg: Config, progress: dict[str, Any]) -> None:    cfg.progress_file.parent.mkdir(parents=True, exist_ok=True)    with open(cfg.progress_file, "w", encoding="utf-8") as f:        json.dump(progress, f, indent=2)def _write_manifests(    cfg: Config,    processed: list[dict],    modalities: dict[str, dict],    labels: dict[str, str],    extra_data: dict[str, dict],    mapping: dict[str, str],    label_groups: dict[str, str],) -> None:    """Write manifest.json and manifest.csv — no real IDs anywhere."""    out_dir = cfg.validated_dir    out_dir.mkdir(parents=True, exist_ok=True)    reverse_map = {v: k for k, v in mapping.items()}    records = []    for item in processed:        pseudo = item["pseudo_id"]        real_id = reverse_map.get(pseudo, "")        mod_info = modalities.get(real_id, {})        birads = labels.get(real_id, "UNKNOWN")        lbl_grp = get_label_group(birads, label_groups)        ex = extra_data.get(real_id, {})        records.append({            "pseudo_id": pseudo,            "n_us": item["n_us"],            "n_mammo": item["n_mammo"],            "us_paths": item["us_paths"],            "mammo_paths": item["mammo_paths"],            "modality_status": mod_info.get("status", "unknown"),            "has_report": mod_info.get("has_report", False),            "birads": birads,            "label_group": lbl_grp,            # Extra TSV columns (no real ID)            "binary_label": ex.get("Binary Label", ""),            "pathology_label": ex.get("Pathology Label", ""),        })    # JSON    with open(out_dir / "manifest.json", "w", encoding="utf-8") as f:        json.dump(records, f, indent=2, ensure_ascii=False)    # CSV    fieldnames = [        "pseudo_id", "n_us", "n_mammo", "us_paths", "mammo_paths",        "modality_status", "has_report", "birads", "label_group",        "binary_label", "pathology_label",    ]    with open(out_dir / "manifest.csv", "w", newline="", encoding="utf-8") as f:        writer = csv.DictWriter(f, fieldnames=fieldnames)        writer.writeheader()        for r in records:            row = dict(r)            row["us_paths"] = ";".join(r["us_paths"])            row["mammo_paths"] = ";".join(r["mammo_paths"])            writer.writerow(row)    logger.info("Manifests written → %s", out_dir)# ──────────────────────────── commands ───────────────────────────────────────def cmd_run(cfg: Config, limit: int | None) -> None:    """Full pipeline."""    from tqdm import tqdm    from .anonymize import build_mapping, save_private_mapping    from .integrity import check_integrity    from .modality import classify_patients, filter_for_training, modality_summary    from .plots import generate_all_plots    from .preprocess import process_patient    from .profiling import profile_dataset, save_profile    from .report import build_report, save_report    from .reports import process_reports    from .splits import make_splits, save_splits    from .validation import validate_labels    from .zip_io import scan_source, auto_find_dataset, auto_find_zip, iter_patients    # ── Directories    for d in [cfg.work_dir, cfg.validated_dir, cfg.mammo_dir,              cfg.us_dir, cfg.reports_dir, cfg.plots_dir]:        d.mkdir(parents=True, exist_ok=True)    progress = _load_progress(cfg)    # ── Step 1: Auto-detect dataset (ZIP or extracted directory)    if "patients" not in progress:        kaggle_input = Path("/kaggle/input")        search_root = kaggle_input if kaggle_input.exists() else Path(".")        # Priority 1: zip_path from config        zip_path = cfg.zip_path if cfg.zip_path and Path(cfg.zip_path).is_file() else None        # Priority 2: auto-find ZIP in /kaggle/input (like notebook1 finds TSV)        if zip_path is None:            zip_path = auto_find_zip(search_root)        # Priority 3: auto-find extracted directory        data_root = None        tsv_auto = None        if zip_path is None:            data_root, tsv_auto = auto_find_dataset(search_root)            if data_root is None:                raise FileNotFoundError(                    "Could not find BCMID dataset. "                    "Import BCMID.zip or the BCMID folder into Kaggle, "                    "or set zip_path in config.yaml."                )        # Use auto-detected TSV if config path not valid        if tsv_auto and (not cfg.tsv_path or not Path(cfg.tsv_path).exists()):            cfg.tsv_path = tsv_auto            logger.info("Auto-detected TSV: %s", tsv_auto)        if zip_path:            cfg.zip_path = zip_path            logger.info("Source: ZIP -> %s", zip_path)        else:            logger.info("Source: directory -> %s", data_root)        all_patients_full = scan_source(zip_path=zip_path, data_root=data_root)        n_total_zip = len(all_patients_full)        patients = all_patients_full        if limit:            patients = dict(list(patients.items())[:limit])            logger.info("--limit %d: keeping %d patients (total: %d)", limit, len(patients), n_total_zip)        progress["patients"] = patients        progress["n_total_zip"] = n_total_zip        _save_progress(cfg, progress)    else:        patients = progress["patients"]        n_total_zip = progress.get("n_total_zip", len(patients))        if limit:            patients = dict(list(patients.items())[:limit])        logger.info("Loaded %d patients from progress cache (total: %d)", len(patients), n_total_zip)    # ── Step 2: Validate labels    logger.info("Validating labels …")    val_result = validate_labels(cfg, patients)    labels = val_result["labels"]    extra_data = val_result.get("extra_data", {})    logger.info(        "TSV columns: %s | ID col: '%s' | Label col: '%s' | Extra: %s",        val_result["tsv_columns"], val_result["id_column"],        val_result["label_column"], val_result.get("extra_columns", []),    )    # ── Step 3: Integrity    logger.info("Checking image integrity …")    integrity_result = check_integrity(cfg, patients)    # ── Step 4: Modality classification (images only, not report)    logger.info("Classifying modalities …")    modalities = classify_patients(cfg, patients, integrity_result["valid_members"])    multimodal_ids, partial_ids = filter_for_training(modalities)    mod_counts = modality_summary(modalities)    # ── Step 5: Profiling    logger.info("Profiling dataset ...")    profile = profile_dataset(cfg, patients, labels, n_total_zip_patients=n_total_zip)    save_profile(profile, cfg.validated_dir)    # ── Step 6: Anonymization    logger.info("Building anonymization mapping …")    all_pids = list(patients.keys())    mapping = build_mapping(all_pids)    save_private_mapping(mapping, cfg.mapping_file)    # ── Step 7: Preprocess images (idempotent, ZIP + dir mode)    logger.info("Preprocessing images …")    processed_results: list[dict] = []    _zip_for_iter = cfg.zip_path if (cfg.zip_path and Path(cfg.zip_path).is_file()) else None    for pid, inv, zf in tqdm(        iter_patients(_zip_for_iter, patients),        desc="Preprocess", unit="patient", total=len(all_pids)    ):        pseudo = mapping[pid]        mod_info = modalities[pid]        first_us = cfg.us_dir / f"{pseudo}_us_0.png"        first_mammo = cfg.mammo_dir / f"{pseudo}_mammo_0.png"        has_us_done = (mod_info["n_valid_us"] == 0) or first_us.exists()        has_mammo_done = (mod_info["n_valid_mammo"] == 0) or first_mammo.exists()        if has_us_done and has_mammo_done:            us_paths = sorted(str(p.relative_to(cfg.validated_dir))                              for p in cfg.us_dir.glob(f"{pseudo}_us_*.png"))            mammo_paths = sorted(str(p.relative_to(cfg.validated_dir))                                 for p in cfg.mammo_dir.glob(f"{pseudo}_mammo_*.png"))            processed_results.append({                "pseudo_id": pseudo,                "us_paths": us_paths,                "mammo_paths": mammo_paths,                "n_us": len(us_paths),                "n_mammo": len(mammo_paths),            })            continue        result = process_patient(pid, pseudo, mod_info, cfg, zf)        processed_results.append(result)    logger.info("Extracting PDF reports …")    reports_stats = process_reports(        patients=patients,        mapping=mapping,        zip_path=cfg.zip_path,        reports_dir=cfg.reports_dir,    )    logger.info(        "Label leakage risk: %.1f%% of reports mention BI-RADS",        reports_stats["label_leakage_risk_pct"],    )    # ── Step 9: Write manifests (with label_group, no real IDs)    logger.info("Writing manifests …")    _write_manifests(cfg, processed_results, modalities, labels,                     extra_data, mapping, DEFAULT_LABEL_GROUPS)    # ── Step 10: Splits (stratified by label_group)    pseudo_multimodal = [mapping[pid] for pid in multimodal_ids if pid in mapping]    pseudo_labels = {mapping[pid]: labels.get(pid, "UNKNOWN")                     for pid in all_pids if pid in mapping}    splits = make_splits(cfg, pseudo_multimodal, pseudo_labels, DEFAULT_LABEL_GROUPS)    save_splits(splits, cfg.validated_dir)    # ── Step 11: Plots    logger.info("Generating plots …")    try:        generate_all_plots(cfg, profile, mod_counts, patients, modalities)    except Exception as exc:        logger.warning("Plot generation failed (non-fatal): %s", exc)    # ── Step 12: Quality report    logger.info("Building quality report …")    preprocessing_params = {        "target_size": cfg.preprocessing.target_size,        "mammo_clahe_clip_limit": cfg.preprocessing.mammo_clahe_clip_limit,        "mammo_clahe_tile_grid": cfg.preprocessing.mammo_clahe_tile_grid,        "us_median_ksize": cfg.preprocessing.us_median_ksize,        "nlm_h": cfg.preprocessing.nlm_h,        "bit_depth": cfg.preprocessing.bit_depth,    }    report = build_report(        profile=profile,        integrity_result=integrity_result,        modality_counts=mod_counts,        validation_result=val_result,        preprocessing_params=preprocessing_params,        n_multimodal_ready=len(multimodal_ids),        reports_stats=reports_stats,    )    save_report(report, cfg.validated_dir)    logger.info("✅ Pipeline complete! Outputs in: %s", cfg.validated_dir)    _print_summary(report, splits, reports_stats)def _print_summary(report: dict, splits: dict, reports_stats: dict) -> None:    sep = "=" * 62    print("\n" + sep)    print("  DATA & QUALITY AGENT -- RESUME FINAL")    print(sep)    print(f"  Dossiers patients (ZIP)      : {report['n_patient_folders']}")    print(f"  Patients annotes (TSV)       : {report['n_patients_in_annotations']}")    print(f"  Ecart detecte                : {report['ecart_patients']}")    print(f"  Prets (multimodal complet)   : {report['patients_ready_multimodal']}")    print(f"  Sans rapport PDF             : {report['patients_without_report_pdf']}")    print(f"  Images corrompues            : {report['corrupted_or_invalid_count']}")    print(f"  Groupes de doublons          : {report['duplicate_files_count']}")    print(f"  Sans label BI-RADS           : {len(report['patients_without_label'])}")    print("-" * 62)    print(f"  BI-RADS distribution         : {report['birads_distribution']}")    print("-" * 62)    print(f"  Rapports PDF extraits (OK)   : {reports_stats.get('reports_extracted_ok', 0)}")    print(f"  Rapports scannes (illisibles): {reports_stats.get('reports_scanned_unreadable', 0)}")    print(f"  Risque label leakage         : {reports_stats.get('label_leakage_risk_pct', 0):.1f}%")    print("-" * 62)    print(f"  Split train/val/test         : {splits['n_train']}/{splits['n_val']}/{splits['n_test']}")    if report.get("labels_without_folder"):        print(f"  [!] Labels sans dossier      : {len(report['labels_without_folder'])}")    print(sep + "\n")def cmd_profile(cfg: Config, limit: int | None) -> None:    from .profiling import profile_dataset, save_profile    from .validation import validate_labels    from .zip_io import scan_source, auto_find_dataset, auto_find_zip    kaggle_input = Path("/kaggle/input")    search_root = kaggle_input if kaggle_input.exists() else Path(".")    zip_path = cfg.zip_path if cfg.zip_path and Path(cfg.zip_path).is_file() else auto_find_zip(search_root)    data_root, tsv_auto = (None, None) if zip_path else auto_find_dataset(search_root)    if tsv_auto and (not cfg.tsv_path or not Path(cfg.tsv_path).exists()):        cfg.tsv_path = tsv_auto    if zip_path:        cfg.zip_path = zip_path    patients = scan_source(zip_path=zip_path, data_root=data_root)    if limit:        patients = dict(list(patients.items())[:limit])    val_result = validate_labels(cfg, patients)    profile = profile_dataset(cfg, patients, val_result["labels"])    save_profile(profile, cfg.validated_dir)    print(f"\n{'='*50}")    print(f"TSV columns     : {val_result['tsv_columns']}")    print(f"ID column       : {val_result['id_column']}")    print(f"Label column    : {val_result['label_column']}")    print(f"Extra columns   : {val_result.get('extra_columns', [])}")    print(f"Image extensions: {profile['image_extensions']}")    print(f"Patients found  : {len(patients)}")    print(f"Patients in TSV : {val_result.get('n_patients_in_annotations', 'N/A')}")    print(f"BI-RADS dist.   : {profile['birads_distribution']}")    print(f"{'='*50}\n")def cmd_splits(cfg: Config) -> None:    from .splits import make_splits, save_splits, DEFAULT_LABEL_GROUPS    manifest_path = cfg.validated_dir / "manifest.json"    if not manifest_path.exists():        print("ERROR: manifest.json not found — run `run` first.")        sys.exit(1)    with open(manifest_path, encoding="utf-8") as f:        records = json.load(f)    multimodal = [r["pseudo_id"] for r in records if r["modality_status"] == "complete"]    labels = {r["pseudo_id"]: r["birads"] for r in records}    splits = make_splits(cfg, multimodal, labels, DEFAULT_LABEL_GROUPS)    save_splits(splits, cfg.validated_dir)    print(f"Splits: train={splits['n_train']}, val={splits['n_val']}, test={splits['n_test']}")def cmd_report(cfg: Config) -> None:    print("Use `run` to regenerate the full report with all pipeline state.")# ──────────────────────────── main ───────────────────────────────────────────def main() -> None:    parser = argparse.ArgumentParser(        prog="python -m dq_agent",        description="Data & Quality Agent — BCMID Multimodal Medical AI",    )    subparsers = parser.add_subparsers(dest="command", required=True)    p_run = subparsers.add_parser("run", help="Run the full pipeline")    p_run.add_argument("--config", default="config.yaml")    p_run.add_argument("--limit", type=int, default=None)    p_profile = subparsers.add_parser("profile", help="Profiling only")    p_profile.add_argument("--config", default="config.yaml")    p_profile.add_argument("--limit", type=int, default=None)    p_splits = subparsers.add_parser("splits", help="Regenerate splits")    p_splits.add_argument("--config", default="config.yaml")    p_report = subparsers.add_parser("report", help="Regenerate report")    p_report.add_argument("--config", default="config.yaml")    args = parser.parse_args()    cfg = load_config(args.config)    setup_logging(cfg)    if args.command == "run":        cmd_run(cfg, args.limit)    elif args.command == "profile":        cmd_profile(cfg, args.limit)    elif args.command == "splits":        cmd_splits(cfg)    elif args.command == "report":        cmd_report(cfg)    else:        parser.print_help()        sys.exit(1)if __name__ == "__main__":    main()

---
## Module : `__main__.py`

Point d'entrée pour `python -m dq_agent`. Appelle simplement `cli.main()`.


In [ ]:
%%writefile dq_agent/__main__.py"""Allows `python -m dq_agent` to invoke the CLI."""from .cli import mainif __name__ == "__main__":    main()

---
## Configuration : `config.yaml`

Ce fichier centralise **tous les paramètres** du pipeline.
Les chemins sont configurés pour Kaggle par défaut.
Modifie-les si tu travailles en local.


---
## Etape 1 : Detection automatique du dataset (comme notebook1)

Cette cellule detecte automatiquement le dataset BCMID importe dans Kaggle,
**exactement comme notebook1** : elle cherche `BCMID_annotations.tsv`
de facon recursive dans `/kaggle/input` et deduit si c'est un ZIP ou un dossier extrait.

**Rien a modifier ici.** La cellule suivante genere `config.yaml` automatiquement.


In [ ]:
import os
from pathlib import Path

# ============================================================
# AUTO-DETECTION DU DATASET — identique a notebook1
# ============================================================

KAGGLE_INPUT  = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

# 1. Recherche recursive du TSV (comme notebook1 fait rglob)
annotation_candidates = list(KAGGLE_INPUT.rglob("BCMID_annotations.tsv"))

if not annotation_candidates:
    raise FileNotFoundError(
        "BCMID_annotations.tsv introuvable dans /kaggle/input.\n"
        "=> Va dans 'Add Data' sur Kaggle et importe le dataset BCMID."
    )

TSV_PATH  = annotation_candidates[0]
DATA_ROOT = TSV_PATH.parent        # dossier contenant le TSV

print(f"TSV trouve     : {TSV_PATH}")
print(f"Racine dataset : {DATA_ROOT}")

# 2. Detecter ZIP ou dossier extrait
zip_candidates = [
    z for z in list(KAGGLE_INPUT.rglob("*.zip"))
    if "bcmid" in z.name.lower() or "breast" in z.name.lower()
]

if zip_candidates:
    ZIP_PATH    = zip_candidates[0]
    SOURCE_MODE = "zip"
    print(f"Mode           : ZIP -> {ZIP_PATH}")
else:
    ZIP_PATH    = None
    SOURCE_MODE = "dir"
    print(f"Mode           : dossier extrait -> {DATA_ROOT}")

WORK_DIR = KAGGLE_WORKING / "bcmid_work"
print(f"Dossier sortie : {WORK_DIR}")


---
## Etape 2 : Generation de config.yaml

Le fichier `config.yaml` est genere automatiquement avec les chemins detectes.


In [ ]:
import yaml

# ============================================================
# GENERATION DE config.yaml avec chemins auto-detectes
# ============================================================

config = {
    "zip_path":    str(ZIP_PATH) if ZIP_PATH else "",
    "tsv_path":    str(TSV_PATH),
    "work_dir":    str(WORK_DIR),
    "private_dir": str(WORK_DIR),
    "preprocessing": {
        "target_size":           512,
        "mammo_clahe_clip_limit": 2.0,
        "mammo_clahe_tile_grid":  8,
        "us_median_ksize":        5,
        "nlm_h":                  10,
        "bit_depth":              8,
    },
    "splits": {
        "train_ratio":  0.70,
        "val_ratio":    0.15,
        "test_ratio":   0.15,
        "random_seed":  42,
    },
    "logging": {
        "level": "INFO",
        "file":  "dq_agent.log",
    },
}

with open("config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("config.yaml genere :")
print(open("config.yaml").read())


---
## Etape 3 : Execution du pipeline complet

Le pipeline detecte automatiquement ZIP ou dossier — **aucune configuration manuelle**.

- Sur Kaggle (GPU T4) : environ **1 a 2 heures**
- **Idempotent** : si interrompu, relancer reprend exactement ou ca s'est arrete


---
## 🚀 Exécution du pipeline complet

La cellule ci-dessous lance le pipeline de bout en bout.
- Sur Kaggle avec GPU T4 : environ **1 à 2 heures**
- Le pipeline est **idempotent** : si interrompu, relancer la cellule reprend là où il s'est arrêté


In [ ]:
# Sel d'anonymisation — changer en production pour un vrai secret
os.environ["DQ_SALT"] = "MonSelSecret_2026"

# Lancer le pipeline complet (auto-detecte ZIP ou dossier)
!python -m dq_agent run --config config.yaml


---
## 📊 Visualisation des résultats

Affichons les graphiques générés et le rapport de qualité.


In [ ]:
from IPython.display import display, Image as IPImage, Markdownfrom pathlib import Pathimport jsonval_dir = Path("/kaggle/working/bcmid_work/validated")# Afficher le rapport Markdownif (val_dir / "quality_report.md").exists():    display(Markdown((val_dir / "quality_report.md").read_text()))# Afficher les graphiquesplots_dir = val_dir / "plots"if plots_dir.exists():    for p in sorted(plots_dir.glob("*.png")):        print(f"\n--- {p.stem} ---")        display(IPImage(filename=str(p), width=700))

In [ ]:
# Afficher le manifest (premiers patients)import pandas as pdmanifest_path = val_dir / "manifest.csv"if manifest_path.exists():    df = pd.read_csv(manifest_path)    print(f"Manifest: {len(df)} patients")    display(df.head(10))# Afficher les splitssplits_path = val_dir / "splits.json"if splits_path.exists():    with open(splits_path) as f:        sp = json.load(f)    print(f"\nSplits: train={sp['n_train']}, val={sp['n_val']}, test={sp['n_test']}")    print(f"Distribution train: {sp['class_distribution']['train']}")

---
## Analyse de la qualite des images (Section 09 — comme notebook1)

Pour chaque image valide, on analyse :
- **Dimensions** (width x height)
- **Mode** (RGBA ou L = niveaux de gris)
- **Intensite moyenne** (mean) et **ecart-type** (std) des pixels

Cela permet de detecter les images anormales avant l'entrainement.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import json

WORK_DIR = Path("/kaggle/working/bcmid_work")
VAL_DIR  = WORK_DIR / "validated"

# ============================================================
# 09. IMAGE QUALITY ANALYSIS (comme notebook1)
# ============================================================

print("=" * 70)
print("09. IMAGE QUALITY ANALYSIS")
print("=" * 70)

# Charger le manifest pour avoir les chemins et pseudo_ids
manifest_path = VAL_DIR / "manifest.json"
with open(manifest_path) as f:
    manifest = json.load(f)

def analyze_image_quality(image_path):
    """Analyse une image : dimensions, mode, mean, std des pixels."""
    result = {
        "exists": False,
        "readable": False,
        "width": None,
        "height": None,
        "mode": None,
        "channels": None,
        "mean_intensity": None,
        "std_intensity": None,
        "error": None
    }
    path = Path(image_path)
    if not path.exists():
        result["error"] = "FILE_NOT_FOUND"
        return result
    result["exists"] = True
    try:
        with Image.open(path) as img:
            result["width"]    = img.width
            result["height"]   = img.height
            result["mode"]     = img.mode
            result["channels"] = len(img.getbands())
            arr = np.array(img.convert("L"), dtype=np.float32)
            result["mean_intensity"] = float(round(arr.mean(), 2))
            result["std_intensity"]  = float(round(arr.std(), 2))
            result["readable"] = True
    except Exception as e:
        result["error"] = str(e)
    return result

# Analyser toutes les images preprocessees
quality_records = []
for patient in manifest:
    pseudo = patient["pseudo_id"]
    birads = patient.get("birads", "?")
    # US images
    for p in patient.get("us_paths", []):
        full_path = VAL_DIR / p
        q = analyze_image_quality(full_path)
        q["pseudo_id"] = pseudo
        q["modality"]  = "Ultrasound"
        q["birads"]    = birads
        q["path"]      = str(full_path)
        quality_records.append(q)
    # Mammo images
    for p in patient.get("mammo_paths", []):
        full_path = VAL_DIR / p
        q = analyze_image_quality(full_path)
        q["pseudo_id"] = pseudo
        q["modality"]  = "Mammogram"
        q["birads"]    = birads
        q["path"]      = str(full_path)
        quality_records.append(q)

quality_df = pd.DataFrame(quality_records)
print(f"\nTotal images analysees : {len(quality_df)}")
print(f"Images lisibles        : {quality_df['readable'].sum()}")
print(f"Images illisibles      : {(~quality_df['readable']).sum()}")


---
## Statistiques par modalite (Mammographie vs Echographie)

Analyse separee des caracteristiques des images selon la modalite.


In [ ]:
from IPython.display import display

# ============================================================
# Statistiques des dimensions par modalite
# ============================================================
readable = quality_df[quality_df["readable"]].copy()

print("\nDimensions des images (width x height) :")
dim_stats = readable.groupby("modality")[["width", "height"]].describe().round(0)
display(dim_stats)

# ============================================================
# Distribution des modes d'image
# ============================================================
print("\nModes d'image par modalite :")
mode_dist = (
    readable.groupby(["modality", "mode"])
    .size()
    .reset_index(name="count")
)
display(mode_dist)

# ============================================================
# Intensite moyenne et ecart-type
# ============================================================
print("\nIntensite des pixels par modalite :")
intensity_stats = readable.groupby("modality")[["mean_intensity", "std_intensity"]].describe().round(2)
display(intensity_stats)

# ============================================================
# Afficher les 10 premieres lignes du tableau complet
# ============================================================
print("\nExemples (10 premieres images) :")
cols = ["pseudo_id", "modality", "birads", "width", "height", "mode", "mean_intensity", "std_intensity"]
display(readable[cols].head(10))


---
## Histogrammes de qualite des images

Visualisation de la distribution des dimensions et intensites
pour detecter des anomalies (images trop petites, trop sombres...).


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Analyse de qualite des images - BCMID", fontsize=14, fontweight="bold")

mammo = readable[readable["modality"] == "Mammogram"]
us    = readable[readable["modality"] == "Ultrasound"]

# 1. Distribution largeur
ax = axes[0, 0]
ax.hist(mammo["width"].dropna(), bins=20, alpha=0.7, label="Mammographie", color="steelblue")
ax.hist(us["width"].dropna(),    bins=20, alpha=0.7, label="Echographie",  color="coral")
ax.set_title("Distribution largeur (width)")
ax.set_xlabel("Pixels")
ax.set_ylabel("Nombre d'images")
ax.legend()

# 2. Distribution hauteur
ax = axes[0, 1]
ax.hist(mammo["height"].dropna(), bins=20, alpha=0.7, label="Mammographie", color="steelblue")
ax.hist(us["height"].dropna(),    bins=20, alpha=0.7, label="Echographie",  color="coral")
ax.set_title("Distribution hauteur (height)")
ax.set_xlabel("Pixels")
ax.legend()

# 3. Modes d'image
ax = axes[0, 2]
mode_counts = readable["mode"].value_counts()
ax.bar(mode_counts.index, mode_counts.values, color=["steelblue", "coral", "green"])
ax.set_title("Modes d'image (RGBA / L)")
ax.set_xlabel("Mode")
ax.set_ylabel("Nombre d'images")
for i, v in enumerate(mode_counts.values):
    ax.text(i, v + 5, str(v), ha="center", fontweight="bold")

# 4. Mean intensity
ax = axes[1, 0]
ax.hist(mammo["mean_intensity"].dropna(), bins=30, alpha=0.7, label="Mammographie", color="steelblue")
ax.hist(us["mean_intensity"].dropna(),    bins=30, alpha=0.7, label="Echographie",  color="coral")
ax.set_title("Intensite moyenne des pixels")
ax.set_xlabel("Valeur (0-255)")
ax.set_ylabel("Nombre d'images")
ax.legend()

# 5. Std intensity
ax = axes[1, 1]
ax.hist(mammo["std_intensity"].dropna(), bins=30, alpha=0.7, label="Mammographie", color="steelblue")
ax.hist(us["std_intensity"].dropna(),    bins=30, alpha=0.7, label="Echographie",  color="coral")
ax.set_title("Ecart-type des pixels (contraste)")
ax.set_xlabel("Valeur")
ax.legend()

# 6. Nombre d'images par patient
ax = axes[1, 2]
imgs_per_patient = readable.groupby("pseudo_id").size()
ax.hist(imgs_per_patient.values, bins=20, color="mediumpurple", edgecolor="white")
ax.set_title("Images par patient")
ax.set_xlabel("Nombre d'images")
ax.set_ylabel("Nombre de patients")
ax.axvline(imgs_per_patient.mean(), color="red", linestyle="--", label=f"Moy={imgs_per_patient.mean():.1f}")
ax.legend()

plt.tight_layout()
plt.savefig(str(VAL_DIR / "plots" / "image_quality_analysis.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Graphique sauvegarde : plots/image_quality_analysis.png")


---
## Pipeline de preprocessing affiche etape par etape

Notre pipeline applique des traitements medicaux specifiques a chaque modalite.
Cette cellule montre **avant / apres** chaque etape pour valider visuellement
la qualite du preprocessing — equivalent de l'affichage des transforms PyTorch
dans notebook1 (Resize, ToTensor, Normalize), mais adapte a l'imagerie medicale.

### Pipeline Echographie (US)
```
Brut --> Niveaux de gris --> Filtre median (anti-speckle) --> Normalisation [0,1] --> Resize 512x512
```
### Pipeline Mammographie
```
Brut --> Niveaux de gris --> Non-Local Means (debruitage) --> CLAHE (contraste) --> Normalisation --> Resize 512x512
```


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image
from pathlib import Path
import json
import random

VAL_DIR  = Path("/kaggle/working/bcmid_work")
VAL_OUT  = VAL_DIR / "validated"

# ============================================================
# PREPROCESSING PIPELINE — Affichage etape par etape
# ============================================================

# --- Fonctions de preprocessing (memes que preprocess.py) ---

def normalize_01(arr):
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-8:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

def median_denoise(arr, ksize=5):
    try:
        import cv2
        u8 = (arr * 255).clip(0, 255).astype(np.uint8)
        return cv2.medianBlur(u8, ksize).astype(np.float32) / 255.0
    except ImportError:
        from scipy.ndimage import median_filter
        return median_filter(arr, size=ksize)

def nlm_denoise(arr, h=10):
    try:
        import cv2
        u8 = (arr * 255).clip(0, 255).astype(np.uint8)
        return cv2.fastNlMeansDenoising(u8, h=h, templateWindowSize=7, searchWindowSize=21).astype(np.float32) / 255.0
    except ImportError:
        return arr  # fallback

def apply_clahe(arr, clip_limit=2.0, tile_grid=8):
    try:
        import cv2
        u8 = (arr * 255).clip(0, 255).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid, tile_grid))
        return clahe.apply(u8).astype(np.float32) / 255.0
    except ImportError:
        return arr

def resize_img(arr, target=512):
    try:
        import cv2
        u8 = (arr * 255).clip(0, 255).astype(np.uint8)
        return cv2.resize(u8, (target, target), interpolation=cv2.INTER_AREA)
    except ImportError:
        pil = Image.fromarray((arr * 255).astype(np.uint8))
        return np.array(pil.resize((target, target), Image.LANCZOS))

# --- Trouver une image brute de chaque modalite ---
manifest_path = VAL_OUT / "manifest.json"
with open(manifest_path) as f:
    manifest_data = json.load(f)

# Recuperer les images preprocessees pour avoir les pseudo_ids
mammo_samples = list((VAL_OUT / "mammo").glob("*.png"))
us_samples    = list((VAL_OUT / "us").glob("*.png"))

random.seed(42)
mammo_proc = str(random.choice(mammo_samples)) if mammo_samples else None
us_proc    = str(random.choice(us_samples))    if us_samples    else None

# ============================================================
# FIGURE 1 : Pipeline Echographie (US)
# ============================================================
if us_proc:
    img_orig = Image.open(us_proc).convert("L")
    arr_orig = np.array(img_orig, dtype=np.float32) / 255.0
    
    # Etapes
    step0 = arr_orig                                   # Brut
    step1 = normalize_01(step0)                        # Normalisation
    step2 = median_denoise(step1, ksize=5)             # Filtre median
    step3 = normalize_01(step2)                        # Renormalisation
    step4 = resize_img(step3, 512)                     # Resize 512x512
    
    steps_us = [
        (step0,  "1. Image brute
(niveaux de gris)"),
        (step1,  "2. Normalisation
[0, 1]"),
        (step2,  "3. Filtre median
(anti-speckle US)"),
        (step3,  "4. Re-normalisation
[0, 1]"),
        (step4,  "5. Resize
512 x 512"),
    ]
    
    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    fig.suptitle("Pipeline preprocessing — ECHOGRAPHIE (Ultrasound)", 
                 fontsize=13, fontweight="bold", color="coral")
    
    for ax, (arr, title) in zip(axes, steps_us):
        ax.imshow(arr if arr.ndim == 2 else arr, cmap="gray")
        ax.set_title(title, fontsize=9)
        ax.axis("off")
        shape = arr.shape[:2]
        mean  = arr.mean() * (255 if arr.max() <= 1.0 else 1)
        std   = arr.std()  * (255 if arr.max() <= 1.0 else 1)
        ax.set_xlabel(f"{shape[1]}x{shape[0]} | mean={mean:.0f} std={std:.0f}", fontsize=8)
    
    # Fleches entre etapes
    for ax in axes[:-1]:
        ax.annotate("", xy=(1.05, 0.5), xytext=(0.95, 0.5),
                    xycoords="axes fraction", textcoords="axes fraction",
                    arrowprops=dict(arrowstyle="->", color="coral", lw=2))
    
    plt.tight_layout()
    plt.savefig(str(VAL_OUT / "plots" / "preprocess_pipeline_us.png"), dpi=120, bbox_inches="tight")
    plt.show()
    print("Sauvegarde : plots/preprocess_pipeline_us.png")

# ============================================================
# FIGURE 2 : Pipeline Mammographie
# ============================================================
if mammo_proc:
    img_orig = Image.open(mammo_proc).convert("L")
    arr_orig = np.array(img_orig, dtype=np.float32) / 255.0
    
    step0 = arr_orig
    step1 = normalize_01(step0)
    step2 = nlm_denoise(step1, h=10)
    step3 = apply_clahe(step2, clip_limit=2.0, tile_grid=8)
    step4 = normalize_01(step3)
    step5 = resize_img(step4, 512)
    
    steps_mammo = [
        (step0,  "1. Image brute
(niveaux de gris)"),
        (step1,  "2. Normalisation
[0, 1]"),
        (step2,  "3. Non-Local Means
(debruitage NLM)"),
        (step3,  "4. CLAHE
(contraste local)"),
        (step4,  "5. Re-normalisation
[0, 1]"),
        (step5,  "6. Resize
512 x 512"),
    ]
    
    fig, axes = plt.subplots(1, 6, figsize=(26, 5))
    fig.suptitle("Pipeline preprocessing — MAMMOGRAPHIE", 
                 fontsize=13, fontweight="bold", color="steelblue")
    
    for ax, (arr, title) in zip(axes, steps_mammo):
        ax.imshow(arr if arr.ndim == 2 else arr, cmap="gray")
        ax.set_title(title, fontsize=9)
        ax.axis("off")
        shape = arr.shape[:2]
        mean  = arr.mean() * (255 if arr.max() <= 1.0 else 1)
        std   = arr.std()  * (255 if arr.max() <= 1.0 else 1)
        ax.set_xlabel(f"{shape[1]}x{shape[0]} | mean={mean:.0f} std={std:.0f}", fontsize=8)
    
    plt.tight_layout()
    plt.savefig(str(VAL_OUT / "plots" / "preprocess_pipeline_mammo.png"), dpi=120, bbox_inches="tight")
    plt.show()
    print("Sauvegarde : plots/preprocess_pipeline_mammo.png")

# ============================================================
# Tableau recapitulatif des transforms (comme notebook1)
# ============================================================
print()
print("=" * 65)
print("PIPELINE PREPROCESSING — RESUME (equivalent PyTorch transforms)")
print("=" * 65)
print()
print("ECHOGRAPHIE (US) :")
print("  Step 1 : Grayscale conversion (L mode)")
print("  Step 2 : Normalize [0, 1]")
print("  Step 3 : Median filter (ksize=5)  — anti-speckle noise")
print("  Step 4 : Re-normalize [0, 1]")
print("  Step 5 : Resize(512, 512)  [INTER_AREA]")
print()
print("MAMMOGRAPHIE :")
print("  Step 1 : Grayscale conversion (L mode)")
print("  Step 2 : Normalize [0, 1]")
print("  Step 3 : Non-Local Means (h=10)   — debruitage avance")
print("  Step 4 : CLAHE (clip=2.0, grid=8) — contraste local")
print("  Step 5 : Re-normalize [0, 1]")
print("  Step 6 : Resize(512, 512)  [INTER_AREA]")
print()
print("  => Sortie : PNG 8 bits, 512x512, niveaux de gris")
print("=" * 65)


---
## Exemples visuels d'images preprocessees

Affichage de quelques mammographies et echographies pour verifier
visuellement la qualite du preprocessing (CLAHE, filtre median, normalisation).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import random

# ============================================================
# Afficher 6 images : 3 mammo + 3 echo
# ============================================================

mammo_paths = readable[readable["modality"] == "Mammogram"]["path"].dropna().tolist()
us_paths    = readable[readable["modality"] == "Ultrasound"]["path"].dropna().tolist()

random.seed(42)
sample_mammo = random.sample(mammo_paths, min(3, len(mammo_paths)))
sample_us    = random.sample(us_paths,    min(3, len(us_paths)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Exemples d'images preprocessees (512x512, niveau de gris)", fontsize=13, fontweight="bold")

for col, path in enumerate(sample_mammo):
    try:
        img = Image.open(path).convert("L")
        arr = np.array(img)
        axes[0, col].imshow(arr, cmap="gray")
        axes[0, col].set_title(f"Mammo ({arr.shape[0]}x{arr.shape[1]})", fontsize=10)
        axes[0, col].axis("off")
        axes[0, col].text(5, 25, f"mean={arr.mean():.0f} std={arr.std():.0f}",
                          color="yellow", fontsize=8, fontweight="bold")
    except Exception as e:
        axes[0, col].set_title(f"Erreur: {e}")
        axes[0, col].axis("off")

for col, path in enumerate(sample_us):
    try:
        img = Image.open(path).convert("L")
        arr = np.array(img)
        axes[1, col].imshow(arr, cmap="gray")
        axes[1, col].set_title(f"Echo ({arr.shape[0]}x{arr.shape[1]})", fontsize=10)
        axes[1, col].axis("off")
        axes[1, col].text(5, 25, f"mean={arr.mean():.0f} std={arr.std():.0f}",
                          color="yellow", fontsize=8, fontweight="bold")
    except Exception as e:
        axes[1, col].set_title(f"Erreur: {e}")
        axes[1, col].axis("off")

# Labels lignes
axes[0, 0].set_ylabel("Mammographie", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("Echographie",  fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(str(VAL_DIR / "plots" / "sample_images.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Graphique sauvegarde : plots/sample_images.png")


---
## Sauvegarde des resultats finaux

Tous les fichiers produits par le pipeline sont listes ici.


In [ ]:
import os
import json
from pathlib import Path

VAL_DIR = Path("/kaggle/working/bcmid_work/validated")

# ============================================================
# 12. SAVE OUTPUTS — Resume des fichiers produits
# ============================================================

print("=" * 70)
print("12. FICHIERS PRODUITS PAR LE PIPELINE")
print("=" * 70)

files_to_check = {
    "Manifest JSON"       : VAL_DIR / "manifest.json",
    "Manifest CSV"        : VAL_DIR / "manifest.csv",
    "Splits JSON"         : VAL_DIR / "splits.json",
    "Rapport qualite JSON": VAL_DIR / "quality_report.json",
    "Rapport qualite MD"  : VAL_DIR / "quality_report.md",
    "Mapping PRIVE"       : Path("/kaggle/working/bcmid_work/id_mapping_PRIVATE.csv"),
}

print()
for name, path in files_to_check.items():
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  [OK] {name:25s} : {path.name} ({size_kb:.1f} KB)")
    else:
        print(f"  [--] {name:25s} : ABSENT")

# Compter les images
mammo_dir = VAL_DIR / "mammo"
us_dir    = VAL_DIR / "us"
rep_dir   = VAL_DIR / "reports"
plots_dir = VAL_DIR / "plots"

n_mammo   = len(list(mammo_dir.glob("*.png"))) if mammo_dir.exists() else 0
n_us      = len(list(us_dir.glob("*.png")))    if us_dir.exists()    else 0
n_reports = len(list(rep_dir.glob("*.txt")))   if rep_dir.exists()   else 0
n_plots   = len(list(plots_dir.glob("*.png"))) if plots_dir.exists() else 0

print()
print(f"  Mammographies preprocessees : {n_mammo} images PNG")
print(f"  Echographies preprocessees  : {n_us} images PNG")
print(f"  Rapports PDF extraits       : {n_reports} fichiers texte")
print(f"  Graphiques                  : {n_plots} fichiers PNG")

# Charger et afficher splits
splits_path = VAL_DIR / "splits.json"
if splits_path.exists():
    with open(splits_path) as f:
        sp = json.load(f)
    print()
    print(f"  Split train : {sp['n_train']} patients")
    print(f"  Split val   : {sp['n_val']} patients")
    print(f"  Split test  : {sp['n_test']} patients")
    print(f"  Distribution train : {sp['class_distribution']['train']}")

# Sauvegarder le tableau qualite des images
quality_csv = VAL_DIR / "image_quality_stats.csv"
if "quality_df" in dir():
    quality_df.to_csv(quality_csv, index=False)
    print(f"\n  [OK] Tableau qualite images : {quality_csv.name}")

print()
print("=" * 70)
print("Pipeline complet. Dataset pret pour l'Agent Vision (CNN/ViT).")
print("=" * 70)


---
## 🧪 Tests Automatisés (37 tests pytest)

Les tests utilisent des **données synthétiques** (créées dans les fixtures) pour valider :
- Détection d'images corrompues et uniformes
- Détection de doublons
- Classification des modalités
- Normalisation des labels BI-RADS (`'4A'`, `'BI-RADS 4A'` → `'4A'`)
- Détection d'IDs dupliqués dans le TSV
- Anonymisation déterministe (même sel → même pseudo)
- Étanchéité des splits (aucun patient dans deux splits)
- Prétraitement (forme de sortie, range des pixels, fichiers créés)
- Génération du rapport de qualité (clés requises, fichier markdown)


In [ ]:
%%writefile tests/conftest.py"""Shared pytest fixtures: synthetic mini-dataset (in-memory ZIP + TSV)."""from __future__ import annotationsimport ioimport zipfilefrom pathlib import Pathimport numpy as npimport pytestfrom PIL import Imagedef _make_png_bytes(w: int = 64, h: int = 64, value: int | None = None) -> bytes:    """Create a valid PNG image as bytes."""    if value is not None:        arr = np.full((h, w), value, dtype=np.uint8)    else:        rng = np.random.default_rng(42)        arr = rng.integers(10, 240, (h, w), dtype=np.uint8)    buf = io.BytesIO()    Image.fromarray(arr, mode="L").save(buf, format="PNG")    return buf.getvalue()def _make_corrupt_bytes() -> bytes:    return b"NOT_A_VALID_IMAGE_FILE_XYZ"@pytest.fixture()def synthetic_zip(tmp_path: Path) -> Path:    """    Build a synthetic ZIP with:      - P001y19 : US (3 valid) + Mammo (2 valid) + report PDF  → complete      - P002y20 : US (1 valid) + Mammo (1 valid) + no report   → complete (report optional)      - P003y21 : US only (2 valid)                            → us_only      - P004y22 : US (1 corrupt + 1 uniform) + Mammo (1 valid) → partial    """    zip_path = tmp_path / "test_dataset.zip"    pdf_bytes = b"%PDF-1.4 fake pdf content"    with zipfile.ZipFile(zip_path, "w") as zf:        # P001: complete        for i in range(3):            zf.writestr(f"P001y19/Ultrasound/img_us_{i}.png", _make_png_bytes())        for i in range(2):            zf.writestr(f"P001y19/Mammogram/img_mam_{i}.png", _make_png_bytes())        zf.writestr("P001y19/P001y19_Report.pdf", pdf_bytes)        # P002: complete, no report        zf.writestr("P002y20/Ultrasound/img_us_0.png", _make_png_bytes())        zf.writestr("P002y20/Mammogram/img_mam_0.png", _make_png_bytes())        # P003: us_only        for i in range(2):            zf.writestr(f"P003y21/Ultrasound/img_us_{i}.png", _make_png_bytes())        # P004: 1 corrupt US + 1 uniform US + 1 valid mammo        zf.writestr("P004y22/Ultrasound/img_us_corrupt.png", _make_corrupt_bytes())        zf.writestr("P004y22/Ultrasound/img_us_uniform.png", _make_png_bytes(value=128))        zf.writestr("P004y22/Mammogram/img_mam_0.png", _make_png_bytes())    return zip_path@pytest.fixture()def synthetic_tsv(tmp_path: Path) -> Path:    """TSV with labels including malformed and duplicate entries."""    tsv_path = tmp_path / "annotations.tsv"    lines = [        "patient_id\tbirads_label",        "P001y19\t4A",        "P002y20\tBI-RADS 3",        "P003y21\t4b",         # malformed casing → normalized to 4B        "P004y22\t4A",        "P005y23\t2",           # label without folder        "P001y19\t5",           # DUPLICATE ID    ]    tsv_path.write_text("\n".join(lines), encoding="utf-8")    return tsv_path@pytest.fixture()def config_for_tests(tmp_path: Path, synthetic_zip: Path, synthetic_tsv: Path):    """Config object pointing to test fixtures."""    from dq_agent.config import Config, IntegrityConfig, ModalityConfig, PreprocessingConfig, SplitsConfig    cfg = Config(        zip_path=synthetic_zip,        tsv_path=synthetic_tsv,        work_dir=tmp_path / "work",        private_dir=tmp_path / "work",    )    cfg.integrity = IntegrityConfig(min_dimension=8, std_threshold=1.0)    cfg.preprocessing = PreprocessingConfig(target_size=64)    cfg.modality_policy = ModalityConfig(require_report=False)    cfg.splits = SplitsConfig(random_seed=0, rare_class_threshold=2)    return cfg

In [ ]:
%%writefile tests/test_anonymize.py"""Tests for anonymization module."""from __future__ import annotationsimport pytestfrom dq_agent.anonymize import pseudonymise, build_mappingdef test_pseudonymise_deterministic():    """Same inputs must always produce the same pseudonym."""    salt = "test_salt_xyz"    pid = "P001y19"    assert pseudonymise(pid, salt) == pseudonymise(pid, salt)def test_pseudonymise_format():    """Pseudonym must start with 'P' and have 11 chars total."""    p = pseudonymise("any_id", "salt")    assert p.startswith("P")    assert len(p) == 11  # P + 10 hex charsdef test_real_id_not_in_pseudo():    """Real patient ID must NOT appear in the pseudonym."""    pid = "P001y19"    pseudo = pseudonymise(pid, "salt")    assert pid not in pseudodef test_mapping_no_collisions():    """All pseudonyms in the mapping must be unique."""    pids = [f"patient_{i:04d}" for i in range(100)]    mapping = build_mapping(pids, salt="unique_salt")    pseudos = list(mapping.values())    assert len(pseudos) == len(set(pseudos)), "Collision in pseudonym mapping"def test_mapping_covers_all_ids():    """Every input ID must have a pseudonym."""    pids = ["A", "B", "C"]    mapping = build_mapping(pids, salt="s")    assert set(mapping.keys()) == set(pids)def test_private_mapping_saved(tmp_path):    """Private mapping file must be created at the right path."""    from dq_agent.anonymize import save_private_mapping, load_private_mapping    mapping = {"real_001": "Pabc1234567", "real_002": "Pxyz9876543"}    path = tmp_path / "id_mapping_PRIVATE.csv"    save_private_mapping(mapping, path)    assert path.exists()    loaded = load_private_mapping(path)    assert loaded == mapping

In [ ]:
%%writefile tests/test_integrity.py"""Tests for integrity module."""from __future__ import annotationsimport pytestfrom dq_agent.integrity import check_integrityfrom dq_agent.zip_io import scan_zipdef test_corrupt_image_detected(config_for_tests):    """Corrupted PNG must appear in corrupted_or_invalid list."""    patients = scan_zip(config_for_tests.zip_path)    result = check_integrity(config_for_tests, patients)    corrupted_members = [c["member"] for c in result["corrupted_or_invalid"]]    assert any("corrupt" in m for m in corrupted_members), (        f"Expected corrupt image in results, got: {corrupted_members}"    )def test_uniform_image_detected(config_for_tests):    """Uniform image (constant pixel value) must be flagged."""    patients = scan_zip(config_for_tests.zip_path)    result = check_integrity(config_for_tests, patients)    corrupted_members = [c["member"] for c in result["corrupted_or_invalid"]]    assert any("uniform" in m for m in corrupted_members), (        f"Expected uniform image in results, got: {corrupted_members}"    )def test_valid_images_pass(config_for_tests):    """Valid PNG images must appear in valid_members."""    patients = scan_zip(config_for_tests.zip_path)    result = check_integrity(config_for_tests, patients)    assert len(result["valid_members"]) >= 7  # P001 (5) + P002 (2) at minimumdef test_duplicate_detection(tmp_path):    """Identical file content must be reported as duplicate."""    import io, zipfile    from PIL import Image    import numpy as np    from dq_agent.config import Config, IntegrityConfig    same_content = b"\x89PNG\r\n\x1a\n" + b"\x00" * 100    # Create a proper PNG    buf = io.BytesIO()    arr = np.random.randint(20, 200, (64, 64), dtype=np.uint8)    Image.fromarray(arr, "L").save(buf, "PNG")    png_bytes = buf.getvalue()    zip_path = tmp_path / "dup.zip"    with zipfile.ZipFile(zip_path, "w") as zf:        zf.writestr("PAT1y18/Ultrasound/img1.png", png_bytes)        zf.writestr("PAT1y18/Mammogram/img2.png", png_bytes)  # same content = duplicate    cfg = Config(zip_path=zip_path, tsv_path=tmp_path / "none.tsv",                 work_dir=tmp_path, private_dir=tmp_path)    cfg.integrity = IntegrityConfig(min_dimension=8, std_threshold=0.5)    patients = scan_zip(zip_path)    result = check_integrity(cfg, patients)    assert result["duplicate_count"] >= 1

In [ ]:
%%writefile tests/test_modality.py"""Tests for modality classification."""from __future__ import annotationsimport pytestfrom dq_agent.integrity import check_integrityfrom dq_agent.modality import classify_patients, filter_for_trainingfrom dq_agent.zip_io import scan_zipdef test_us_only_patient(config_for_tests):    """P003y21 has only US images — must be classified as us_only."""    patients = scan_zip(config_for_tests.zip_path)    integrity = check_integrity(config_for_tests, patients)    modalities = classify_patients(config_for_tests, patients, integrity["valid_members"])    assert "P003y21" in modalities    assert modalities["P003y21"]["status"] == "us_only"def test_complete_patient(config_for_tests):    """P001y19 has US + Mammo — must be complete."""    patients = scan_zip(config_for_tests.zip_path)    integrity = check_integrity(config_for_tests, patients)    modalities = classify_patients(config_for_tests, patients, integrity["valid_members"])    assert modalities["P001y19"]["status"] == "complete"def test_filter_for_training(config_for_tests):    """Multimodal list must not include us_only patients."""    patients = scan_zip(config_for_tests.zip_path)    integrity = check_integrity(config_for_tests, patients)    modalities = classify_patients(config_for_tests, patients, integrity["valid_members"])    multimodal, partial = filter_for_training(modalities)    assert "P003y21" not in multimodal    assert "P003y21" in partial

In [ ]:
%%writefile tests/test_preprocess.py"""Tests for preprocessing module."""from __future__ import annotationsimport ioimport numpy as npimport pytestfrom PIL import Imagedef _make_arr(w=64, h=64, seed=0) -> np.ndarray:    rng = np.random.default_rng(seed)    return rng.integers(20, 235, (h, w), dtype=np.uint8).astype(np.float32) / 255.0def test_preprocess_us_output_shape(config_for_tests):    """US preprocessing must output target_size × target_size array."""    from dq_agent.preprocess import preprocess_us    arr = _make_arr()    result = preprocess_us(arr, config_for_tests)    assert result.shape == (64, 64), f"Expected (64,64), got {result.shape}"def test_preprocess_mammo_output_shape(config_for_tests):    """Mammography preprocessing must output target_size × target_size array."""    from dq_agent.preprocess import preprocess_mammo    arr = _make_arr()    result = preprocess_mammo(arr, config_for_tests)    assert result.shape == (64, 64)def test_preprocess_output_uint8_range(config_for_tests):    """Preprocessed arrays must have values in [0, 255] (uint8)."""    from dq_agent.preprocess import preprocess_us    arr = _make_arr()    result = preprocess_us(arr, config_for_tests)    assert result.min() >= 0    assert result.max() <= 255def test_normalize_01():    """_normalize_01 must map min→0 and max→1."""    from dq_agent.preprocess import _normalize_01    arr = np.array([10.0, 50.0, 100.0])    normed = _normalize_01(arr)    assert abs(normed.min()) < 1e-6    assert abs(normed.max() - 1.0) < 1e-6def test_process_patient_saves_files(config_for_tests):    """process_patient must create PNG files on disk."""    import zipfile    from dq_agent.zip_io import scan_zip    from dq_agent.integrity import check_integrity    from dq_agent.modality import classify_patients    from dq_agent.preprocess import process_patient    cfg = config_for_tests    patients = scan_zip(cfg.zip_path)    integrity = check_integrity(cfg, patients)    modalities = classify_patients(cfg, patients, integrity["valid_members"])    pid = "P001y19"    pseudo = "Ptest12345a"    cfg.mammo_dir.mkdir(parents=True, exist_ok=True)    cfg.us_dir.mkdir(parents=True, exist_ok=True)    with zipfile.ZipFile(cfg.zip_path, "r") as zf:        result = process_patient(pid, pseudo, modalities[pid], cfg, zf)    assert result["n_us"] > 0    assert result["n_mammo"] > 0    for p in result["us_paths"]:        assert (cfg.validated_dir / p).exists()    for p in result["mammo_paths"]:        assert (cfg.validated_dir / p).exists()

In [ ]:
%%writefile tests/test_report.py"""Tests for report module."""from __future__ import annotationsimport jsonfrom pathlib import Pathdef _base_profile():    return {        "n_patient_folders": 332,        "n_patients_in_annotations": 323,        "ecart_patients": 9,        "patients_only_in_folders": ["X1", "X2"],        "patients_only_in_annotations": [],        "image_extensions": {"png": 2001},        "birads_distribution": {"3": 50, "4A": 120, "4B": 80, "5": 73},        "images_without_report": 11,        "us_per_patient": {"min": 1, "max": 16, "mean": 3.2, "std": 1.1},        "mammo_per_patient": {"min": 1, "max": 4, "mean": 2.9, "std": 0.6},        "per_patient": [],    }def _base_integrity():    return {        "corrupted_count": 3,        "corrupted_sample": [{"member": "a/b.png", "reason": "corrupt"}],        "duplicate_count": 0,        "duplicate_files": [],    }def _base_validation():    return {        "n_patients_in_annotations": 323,        "patients_without_label": ["P_X1"],        "labels_without_folder": ["P_Y2"],        "duplicate_ids_in_annotations": [],        "malformed_labels": [],        "tsv_columns": ["patient_id", "birads"],        "id_column": "patient_id",        "label_column": "birads",        "issues": [],    }def test_report_contains_required_keys(tmp_path):    from dq_agent.report import build_report, save_report    report = build_report(        profile=_base_profile(),        integrity_result=_base_integrity(),        modality_counts={"complete": 321, "us_only": 0},        validation_result=_base_validation(),        preprocessing_params={"target_size": 512},        n_multimodal_ready=321,    )    required_keys = [        "n_patient_folders", "n_patients_in_annotations", "ecart_patients",        "modality_status", "birads_distribution", "corrupted_or_invalid_count",        "duplicate_files_count", "patients_without_label", "labels_without_folder",        "duplicate_ids_in_annotations", "patients_ready_multimodal",        "parametres_preprocessing", "library_versions", "date_generated",    ]    for key in required_keys:        assert key in report, f"Missing required key in report: {key}"def test_markdown_report_created(tmp_path):    from dq_agent.report import build_report, save_report    report = build_report(        profile=_base_profile(),        integrity_result=_base_integrity(),        modality_counts={"complete": 321},        validation_result=_base_validation(),        preprocessing_params={"target_size": 512},        n_multimodal_ready=321,    )    save_report(report, tmp_path)    md_path = tmp_path / "quality_report.md"    json_path = tmp_path / "quality_report.json"    assert md_path.exists()    assert json_path.exists()    # Check MD has tables    content = md_path.read_text(encoding="utf-8")    assert "BI-RADS" in content    assert "|" in content  # has table    # Check JSON is valid    with open(json_path, encoding="utf-8") as f:        loaded = json.load(f)    assert loaded["ecart_patients"] == 9

In [ ]:
%%writefile tests/test_splits.py"""Tests for splits module — no data leakage between train/val/test."""from __future__ import annotationsimport pytestfrom dq_agent.splits import make_splitsdef test_no_patient_in_two_splits(config_for_tests):    """Core invariant: no patient appears in more than one split."""    multimodal = ["P001", "P002", "P003", "P004", "P005", "P006", "P007", "P008"]    labels = {p: "4A" for p in multimodal}    splits = make_splits(config_for_tests, multimodal, labels)    train = set(splits["train"])    val = set(splits["val"])    test = set(splits["test"])    assert train.isdisjoint(val), f"Patients in train AND val: {train & val}"    assert train.isdisjoint(test), f"Patients in train AND test: {train & test}"    assert val.isdisjoint(test), f"Patients in val AND test: {val & test}"def test_all_patients_assigned(config_for_tests):    """Every multimodal patient must appear in exactly one split."""    multimodal = [f"P{i:03d}" for i in range(20)]    labels = {p: "3" for p in multimodal}    splits = make_splits(config_for_tests, multimodal, labels)    all_assigned = set(splits["train"]) | set(splits["val"]) | set(splits["test"])    assert all_assigned == set(multimodal)def test_rare_class_grouping():    """Classes with fewer than threshold patients must be grouped as RARE."""    from dq_agent.config import Config, SplitsConfig    cfg = Config.__new__(Config)    cfg.splits = SplitsConfig(random_seed=42, rare_class_threshold=3)    # Only 2 patients with "0" → rare    multimodal = ["P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "P9", "P10"]    labels = {        "P1": "0", "P2": "0",           # rare (2 < 3)        "P3": "3", "P4": "3", "P5": "3",        "P6": "4A", "P7": "4A", "P8": "4A",        "P9": "4B", "P10": "4B",         # rare (2 < 3)    }    splits = make_splits(cfg, multimodal, labels)    assert "0" in splits["rare_groups_pooled"] or "4B" in splits["rare_groups_pooled"]def test_splits_deterministic():    """Same seed must always produce same splits."""    from dq_agent.config import Config, SplitsConfig    cfg = Config.__new__(Config)    cfg.splits = SplitsConfig(random_seed=99, rare_class_threshold=2)    multimodal = [f"P{i:03d}" for i in range(30)]    labels = {p: "3" for p in multimodal}    s1 = make_splits(cfg, multimodal, labels)    s2 = make_splits(cfg, multimodal, labels)    assert s1["train"] == s2["train"]    assert s1["val"] == s2["val"]    assert s1["test"] == s2["test"]

In [ ]:
%%writefile tests/test_validation.py"""Tests for validation module."""from __future__ import annotationsimport pytestfrom dq_agent.validation import normalize_birads, validate_labelsfrom dq_agent.zip_io import scan_zip@pytest.mark.parametrize("raw, expected", [    ("4A", "4A"),    ("4a", "4A"),    ("BI-RADS 4A", "4A"),    ("bi-rads 4b", "4B"),    ("BIRADS 3", "3"),    ("BI RADS 4C", "4C"),    ("2", "2"),    ("0", "0"),    ("5", "5"),])def test_birads_normalization(raw, expected):    assert normalize_birads(raw) == expecteddef test_duplicate_id_in_tsv_detected(config_for_tests):    """P001y19 appears twice in test TSV — must be reported."""    patients = scan_zip(config_for_tests.zip_path)    result = validate_labels(config_for_tests, patients)    assert "P001y19" in result["duplicate_ids_in_annotations"]def test_label_without_folder_detected(config_for_tests):    """P005y23 exists in TSV but has no folder — must be reported."""    patients = scan_zip(config_for_tests.zip_path)    result = validate_labels(config_for_tests, patients)    assert "P005y23" in result["labels_without_folder"]def test_malformed_label_4b(config_for_tests):    """'4b' (lowercase) must be normalized to '4B' (not flagged as malformed)."""    patients = scan_zip(config_for_tests.zip_path)    result = validate_labels(config_for_tests, patients)    # 4b → 4B is valid    malformed_pids = [m["patient_id"] for m in result["malformed_labels"]]    assert "P003y21" not in malformed_pids    # Label must be normalized    assert result["labels"]["P003y21"] == "4B"def test_unknown_label_for_missing_patient(config_for_tests, tmp_path):    """Patient in folder but not in TSV gets UNKNOWN label."""    # Modify config to use a TSV without P004y22    tsv_path = tmp_path / "partial.tsv"    tsv_path.write_text("patient_id\tbirads_label\nP001y19\t4A\n", encoding="utf-8")    from dq_agent.config import Config    cfg = Config(        zip_path=config_for_tests.zip_path,        tsv_path=tsv_path,        work_dir=tmp_path,        private_dir=tmp_path,    )    patients = scan_zip(cfg.zip_path)    result = validate_labels(cfg, patients)    assert "P004y22" in result["patients_without_label"]    assert result["labels"]["P004y22"] == "UNKNOWN"

### Exécution des tests


In [ ]:
!python -m pytest tests/ -v --tb=short

---
## ✅ Récapitulatif

Le **Data & Quality Agent** a produit dans `bcmid_work/validated/` :

| Sortie | Description |
|---|---|
| `mammo/` | Images mammographiques prétraitées (512×512, CLAHE + NLM) |
| `us/` | Images échographiques prétraitées (512×512, filtre médian) |
| `reports/` | Textes des rapports PDF anonymisés |
| `manifest.json` / `.csv` | Métadonnées par patient (pseudo_id, birads, label_group, binary_label, pathology_label) |
| `splits.json` | Découpage train/val/test stratifié par patient |
| `quality_report.json` / `.md` | Rapport de qualité complet |
| `plots/` | 6 visualisations PNG |

### Anomalies détectées dans le dataset BCMID :
- 🔴 **1 image corrompue** (exclue automatiquement)
- 🟡 **6 groupes de doublons** (tous intra-patient, pas de fuite entre splits)
- 🔴 **96% de label leakage** dans les rapports PDF (les rapports mentionnent le score BI-RADS)
- 🟢 **0 patient sans label**, **0 label sans dossier**, **0 ID dupliqué**

### Prochaine étape : Agent 3 — Multimodal Vision Agent (CNN/ViT + Grad-CAM)
